In [1]:
# -*- coding: utf-8 -*-
"""
Physics-Informed Neural Network (PINN) for Slope Stability — Device 108
========================================================================
Ablation Study: PIML (Full Richards) vs Vanilla MLP (Data-Only Baseline)

  PIML: ∂θ/∂t = ∂/∂z [ K(ψ) · (∂ψ/∂z + 1) ]  ← physics constrained
  ML  : Data loss only (LAM=0, LAM_ZVAR=0)     ← pure data driven

FIXES APPLIED (PIML):
  Fix 1: θ_pred diagnostic + psi outlier clip (2nd–99th percentile)
  Fix 2: Physics loss normalization (unit mismatch সমাধান)
  Fix 3: z-gradient variance penalty (trivial satisfaction সমাধান)
"""
import warnings; warnings.filterwarnings("ignore")
import os, json, datetime
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler  # scaler_y only
from sklearn.metrics import r2_score, mean_squared_error
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Reproducibility ─────────────────────────────────────────
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

OUT = "/content/piml_figures_108_pinn_richards"
os.makedirs(OUT, exist_ok=True)

STYLE = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.labelsize": 11, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 300, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.facecolor": "white",
}
plt.rcParams.update(STYLE)

C = {
    "rain": "#4895EF", "obs": "#E63946", "pred": "#2EC4B6",
    "train": "#3A86FF", "val": "#FF006E", "test": "#FB5607",
    "psi": "#7209B7", "u": "#F77F00", "fos": "#06D6A0",
    "warn": "#EF233C", "grid": "#CCCCCC", "resid": "#480CA8",
    "phys": "#023E8A", "vg1": "#0077B6", "vg2": "#00B4D8",
    "coral": "#FF7F50",
}

def savefig(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  → Saved: {path}")
    return path

def add_panel_label(ax, label, x=-0.08, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top", ha="right")

def light_grid(ax, axis="y"):
    ax.grid(axis=axis, color=C["grid"], lw=0.5, ls="--", alpha=0.7)

# ═══════════════════════════════════════════════════════════
# VAN GENUCHTEN PARAMETERS  (Sandy Clay Loam, Device 108)
# ═══════════════════════════════════════════════════════════
VG = dict(
    theta_r=0.05, theta_s=0.55,
    alpha=5.9, n=1.48, m=1 - 1/1.48,
    Ks=0.0043,           # m/h
)
VG["Ks_ms"] = VG["Ks"] / 3600.0  # m/s

# Collocation depths (m, positive downward)
Z_SENSOR    = 0.30
Z_COLLOC    = np.array([ 0.30, 0.50], dtype=np.float32)

# ═══════════════════════════════════════════════════════════
# VAN GENUCHTEN TORCH UTILITIES
# ═══════════════════════════════════════════════════════════

def vg_Se_t(theta, vg):
    tc = theta.clamp(vg["theta_r"] + 1e-6, vg["theta_s"] - 1e-6)
    return (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])

def vg_psi_t(theta, vg):
    Se = vg_Se_t(theta, vg).clamp(1e-6, 1 - 1e-6)
    h  = (1.0 / vg["alpha"]) * (Se**(-1.0 / vg["m"]) - 1.0)**(1.0 / vg["n"])
    return -h

def vg_K_t(theta, vg):
    Se = vg_Se_t(theta, vg).clamp(1e-6, 1 - 1e-6)
    Kr = Se**0.5 * (1.0 - (1.0 - Se**(1.0 / vg["m"]))**vg["m"])**2
    return vg["Ks_ms"] * Kr

# NumPy equivalents (evaluation only)
def vg_Se_np(theta, vg):
    tc = np.clip(theta, vg["theta_r"]+1e-6, vg["theta_s"]-1e-6)
    return (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])

def vg_psi_np(theta, vg):
    Se = np.clip(vg_Se_np(theta, vg), 1e-6, 1-1e-6)
    return -(1.0/vg["alpha"]) * (Se**(-1.0/vg["m"]) - 1.0)**(1.0/vg["n"])

def vg_Kr_np(theta, vg):
    Se = np.clip(vg_Se_np(theta, vg), 1e-6, 1-1e-6)
    return Se**0.5 * (1-(1-Se**(1/vg["m"]))**vg["m"])**2

def vg_K_np(theta, vg):
    return vg["Ks_ms"] * vg_Kr_np(theta, vg)

# ═══════════════════════════════════════════════════════════
# 1-2.  LOAD & PREPROCESS  (Device 107 only)
# ═══════════════════════════════════════════════════════════
print("\n[Step 1-2] Loading data for Device 108 ...")
df_raw = pd.read_csv("/content/pinn_108_new.csv", low_memory=False)
df_raw = df_raw.dropna(subset=["timestamp", "devID", "soil", "rain"])
df_raw["devID"]     = df_raw["devID"].astype(int)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], dayfirst=False, errors="coerce")
df_raw = df_raw.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

d108 = df_raw[df_raw["devID"] == 108].copy().reset_index(drop=True)
d108["t_min"] = d108["timestamp"].dt.floor("min")
df = d108[["t_min","soil","rain"]].rename(
    columns={"t_min":"timestamp","soil":"theta","rain":"rain"}).copy()
df = df.sort_values("timestamp").reset_index(drop=True)
df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

df = df.dropna(subset=["rain","theta","t_sec"]).reset_index(drop=True)
df = df.iloc[::2].reset_index(drop=True)
print(f"  Rows after subsample: {len(df):,}")

LAG = 15
n   = len(df)
n_tr  = int(0.80 * n)
n_val = int(0.90 * n)

df_tr  = df.iloc[:n_tr].copy().reset_index(drop=True)
df_val = df.iloc[n_tr:n_val].copy().reset_index(drop=True)
df_te  = df.iloc[n_val:].copy().reset_index(drop=True)

ROLL_W = 5
for _d in [df_tr, df_val, df_te]:
    _d["theta"] = _d["theta"].rolling(ROLL_W, center=False, min_periods=1).median()

def build_features(d, lag=1):
    X_rows, y_rows = [], []
    for i in range(lag, len(d)):
        lags = [d["theta"].iloc[i-k] for k in range(1, lag+1)]
        X_rows.append([d["rain"].iloc[i]] + lags)
        y_rows.append(d["theta"].iloc[i])
    X      = np.array(X_rows, dtype=np.float32)
    y      = np.array(y_rows, dtype=np.float32).reshape(-1,1)
    rain_  = d["rain"].values[lag:].astype(np.float32)
    theta_ = d["theta"].values[lag:].astype(np.float32)
    t_sec_ = d["t_sec"].values[lag:].astype(np.float32)
    ts_    = d["timestamp"].values[lag:]
    return X, y, rain_, theta_, t_sec_, ts_

X_tr_raw,  y_tr_raw,  rain_tr,  theta_tr,  t_sec_tr,  ts_tr  = build_features(df_tr,  LAG)
X_val_raw, y_val_raw, rain_val, theta_val, t_sec_val, ts_val  = build_features(df_val, LAG)
X_te_raw,  y_te_raw,  rain_te,  theta_te,  t_sec_te,  ts_te   = build_features(df_te,  LAG)

# ── Physics-based normalization (replaces MinMaxScaler for X) ──────
# Rain normalized by regional design storm threshold (RAIN_MAX=75 mm/30min);
# lagged θ bounded by Van Genuchten theta_r / theta_s — valid beyond observed range.
RAIN_MAX = 75.0   # mm/30 min — regional upper bound (IDF curve)

def physics_normalize_X(X_raw):
    """Physics-based normalization: rain → RAIN_MAX; lagged θ → VG bounds."""
    X_norm = X_raw.copy()
    X_norm[:, 0]  = X_raw[:, 0] / RAIN_MAX                                           # rain
    X_norm[:, 1:] = (X_raw[:, 1:] - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"])  # lags
    return X_norm.astype(np.float32)

scaler_y = MinMaxScaler().fit(y_tr_raw)  # output scaler unchanged

X_tr  = physics_normalize_X(X_tr_raw)
X_val = physics_normalize_X(X_val_raw)
X_te  = physics_normalize_X(X_te_raw)
y_tr  = scaler_y.transform(y_tr_raw).astype(np.float32)
y_val = scaler_y.transform(y_val_raw).astype(np.float32)
y_te  = scaler_y.transform(y_te_raw).astype(np.float32)

X_np      = np.concatenate([X_tr_raw, X_val_raw, X_te_raw], axis=0)
theta_np  = np.concatenate([theta_tr,  theta_val,  theta_te],  axis=0)
rain_np   = np.concatenate([rain_tr,   rain_val,   rain_te],   axis=0)
t_sec_np  = np.concatenate([t_sec_tr,  t_sec_val,  t_sec_te],  axis=0)
ts_all_np = np.concatenate([ts_tr,     ts_val,     ts_te],     axis=0)
X_norm    = physics_normalize_X(X_np)

T_MIN   = float(t_sec_tr.min())
T_MAX   = float(t_sec_tr.max()) + 1e-8
Z_MAX   = float(Z_COLLOC.max()) + 1e-8

def to_t(a): return torch.tensor(a, dtype=torch.float32).to(DEVICE)

Xt, yt   = to_t(X_tr),  to_t(y_tr)
Xv, yv   = to_t(X_val), to_t(y_val)
Xte, yte = to_t(X_te),  to_t(y_te)

rain_t  = to_t(rain_tr[:, None])
t_sec_t = to_t(t_sec_tr[:, None])
dt_tr   = np.diff(t_sec_tr, prepend=t_sec_tr[0]).clip(min=1.0)
dt_t    = to_t(dt_tr[:, None])

# ═══════════════════════════════════════════════════════════
# 3.  MODEL
# ═══════════════════════════════════════════════════════════
print("\n[Step 3] Building PINN (true autograd Richards) ...")

class RichardsPINN(nn.Module):
    def __init__(self, feat_dim, h=128):
        super().__init__()
        in_dim = feat_dim + 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.Tanh(),
            nn.Linear(h, h),      nn.Tanh(),
            nn.Linear(h, h),      nn.Tanh(),
            nn.Linear(h, h//2),   nn.Tanh(),
            nn.Linear(h//2, 1),   nn.Sigmoid(),
        )
        self.theta_lo = VG["theta_r"] + 0.01
        self.theta_hi = VG["theta_s"] - 0.01

    def forward(self, z_norm, t_norm, feat):
        x   = torch.cat([z_norm, t_norm, feat], dim=1)
        raw = self.net(x)
        theta = self.theta_lo + (self.theta_hi - self.theta_lo) * raw
        return theta

model = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ═══════════════════════════════════════════════════════════
# 4.  PHYSICS LOSS  — [FIX 3] dθ/dz penalty added
# ═══════════════════════════════════════════════════════════

def richards_residual(model, z_norm, t_norm, feat):
    """
    Compute 1D Richards PDE residual + dθ/dz for z-gradient penalty.

    Returns
    -------
    residual   : (B,1)  m³/m³/s
    theta      : (B,1)  predicted θ̂
    dtheta_dz  : (B,1)  ∂θ/∂z  [FIX 3: returned for z-gradient penalty]
    """
    theta = model(z_norm, t_norm, feat)

    dtheta_dt = torch.autograd.grad(
        outputs=theta, inputs=t_norm,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    psi = vg_psi_t(theta, VG)

    dpsi_dz = torch.autograd.grad(
        outputs=psi, inputs=z_norm,
        grad_outputs=torch.ones_like(psi),
        create_graph=True, retain_graph=True
    )[0]

    K = vg_K_t(theta, VG)

    dpsi_dz_phys = dpsi_dz / Z_MAX
    flux = K * (dpsi_dz_phys + 1.0)

    dflux_dz = torch.autograd.grad(
        outputs=flux, inputs=z_norm,
        grad_outputs=torch.ones_like(flux),
        create_graph=True, retain_graph=True
    )[0]

    dflux_dz_phys = dflux_dz / Z_MAX
    dtheta_dt_phys = dtheta_dt / (T_MAX - T_MIN)
    residual = dtheta_dt_phys - dflux_dz_phys

    # ── [FIX 3] ∂θ/∂z — for z-gradient variance penalty ─────
    dtheta_dz = torch.autograd.grad(
        outputs=theta, inputs=z_norm,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    return residual, theta, dtheta_dz   # ← now returns 3 values


# ═══════════════════════════════════════════════════════════
# 5.  COLLOCATION POINT BUILDER
# ═══════════════════════════════════════════════════════════

def make_colloc_batch(t_sec_batch, rain_batch, feat_batch):
    Nz = len(Z_COLLOC)
    B  = t_sec_batch.shape[0]
    z_phys   = np.tile(Z_COLLOC, B).reshape(-1, 1).astype(np.float32)
    z_norm_c = z_phys / Z_MAX
    t_phys_c = np.repeat(t_sec_batch, Nz).reshape(-1, 1).astype(np.float32)
    t_norm_c = (t_phys_c - T_MIN) / (T_MAX - T_MIN)
    feat_c   = np.repeat(feat_batch, Nz, axis=0).astype(np.float32)
    z_c    = torch.tensor(z_norm_c, dtype=torch.float32,
                          device=DEVICE, requires_grad=True)
    t_c    = torch.tensor(t_norm_c, dtype=torch.float32,
                          device=DEVICE, requires_grad=True)
    feat_c = torch.tensor(feat_c,   dtype=torch.float32, device=DEVICE)
    return z_c, t_c, feat_c


# ═══════════════════════════════════════════════════════════
# 6.  TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("\n[Step 4-6] Training PINN with true Richards autograd loss ...")
EPOCHS   = 600
LR       = 5e-4
LAM      = 5
BATCH    = 512
WARMUP   = 100
LAM_ZVAR = 0.1   # [FIX 3] z-gradient variance penalty weight

# ── [FIX 2] Physical scale for residual normalization ───────
THETA_RANGE = VG["theta_s"] - VG["theta_r"]          # 0.50 m³/m³
T_RANGE_SEC = float(T_MAX - T_MIN)                    # seconds
DTHDT_SCALE = THETA_RANGE / T_RANGE_SEC               # typical ∂θ/∂t, m³/m³/s
print(f"  ∂θ/∂t scale for normalization: {DTHDT_SCALE:.4e} m³/m³/s")

y_min_t = torch.tensor(scaler_y.data_min_[0], dtype=torch.float32, device=DEVICE)
y_scl_t = torch.tensor(scaler_y.scale_[0],    dtype=torch.float32, device=DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
mse_loss  = nn.MSELoss()

z_sens_norm = torch.tensor([[Z_SENSOR / Z_MAX]], dtype=torch.float32, device=DEVICE)

t_tr_norm   = ((t_sec_tr  - T_MIN) / (T_MAX - T_MIN)).astype(np.float32)
t_tr_norm_t = to_t(t_tr_norm[:, None])

dataset = TensorDataset(Xt, yt, rain_t, t_tr_norm_t, dt_t,
                        to_t(t_sec_tr[:, None]))
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=False)

data_losses  = []
phys_losses  = []
train_losses = []
val_losses   = []
lr_history   = []

best_val   = np.inf
best_state = None

z_v_rep  = z_sens_norm.expand(Xv.shape[0], -1).detach()
t_v_norm = to_t(((t_sec_val - T_MIN) / (T_MAX - T_MIN))[:, None])

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_data  = 0.0
    ep_phys  = 0.0

    for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
        optimizer.zero_grad()
        B = Xb.shape[0]

        # ── Data loss at sensor depth ─────────────────────────
        z_d = z_sens_norm.expand(B, -1).requires_grad_(True)
        t_d = tb_norm.clone().requires_grad_(True)
        theta_pred = model(z_d, t_d, Xb)

        pred_norm = (theta_pred - y_min_t) * y_scl_t
        loss_d = mse_loss(pred_norm, yb)

        # ── Physics loss at collocation depths ────────────────
        tb_sec_np = tb_sec.detach().cpu().numpy().flatten()
        Xb_np     = Xb.detach().cpu().numpy()
        z_c, t_c, feat_c = make_colloc_batch(tb_sec_np, None, Xb_np)

        # [FIX 3] richards_residual now returns 3 values
        resid, _, dtheta_dz = richards_residual(model, z_c, t_c, feat_c)

        # ── [FIX 2] Normalize residual → dimensionless ────────
        resid_norm = resid / (DTHDT_SCALE + 1e-12)
        loss_p = (resid_norm ** 2).mean()

        # ── [FIX 3] Z-gradient variance penalty ───────────────
        # Reshape to (B, Nz), penalize if all depths have same dθ/dz
        Nz = len(Z_COLLOC)
        dtheta_dz_r = dtheta_dz.view(B, Nz)          # (B, Nz)
        z_var_loss  = -dtheta_dz_r.var(dim=1).mean()  # maximize z-variance

        loss = loss_d + LAM * loss_p + LAM_ZVAR * z_var_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        ep_data += loss_d.item()
        ep_phys += loss_p.item()

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    model.eval()
    with torch.no_grad():
        vl = mse_loss(
            (model(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t,
            yv
        ).item()

    n_batches = len(loader)
    data_losses.append(ep_data / n_batches)
    phys_losses.append(ep_phys / n_batches)
    train_losses.append((ep_data + LAM * ep_phys) / n_batches)
    val_losses.append(vl)
    lr_history.append(current_lr)

    if vl < best_val:
        best_val   = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 100 == 0:
        # ── [FIX 3 diagnostic] λ·Phys / Data ratio ──────────
        ratio = (LAM * phys_losses[-1]) / (data_losses[-1] + 1e-15)
        print(f"  Epoch {epoch:4d} | Data {data_losses[-1]:.5f} | "
              f"Phys(norm) {phys_losses[-1]:.5f} | λ·Phys/Data ratio: {ratio:.3f} | "
              f"Val {vl:.5f} | LR {current_lr:.2e}")

model.load_state_dict(best_state)
print(f"\n  Best val: {best_val:.5f}  @ epoch {int(np.argmin(val_losses))+1}")

def predict_sensor(X_norm_np, t_sec_arr):
    model.eval()
    N = X_norm_np.shape[0]
    z_s  = z_sens_norm.expand(N, -1)
    t_s  = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    Xn   = to_t(X_norm_np)
    with torch.no_grad():
        return model(z_s, t_s, Xn).cpu().numpy().flatten()

# ═══════════════════════════════════════════════════════════
# MODEL SAVE
# ═══════════════════════════════════════════════════════════
print("\n[Model Save] Saving model ...")
model_path = os.path.join(OUT, "piml_slope_model_108_pinn_richards.pt")
torch.save({
    "model_state_dict"  : best_state,
    "model_architecture": {"class": "RichardsPINN", "feat_dim": X_tr.shape[1], "hidden": 128},
    "governing_equation": (
        "1D Full Richards: ∂θ/∂t = ∂/∂z[K(ψ(θ))(∂ψ/∂z + 1)]\n"
        "All derivatives via torch.autograd.grad — no approximation."
    ),
    "hyperparameters": {
        "epochs": EPOCHS, "lr": LR, "lambda_phys": LAM,
        "lambda_zvar": LAM_ZVAR,
        "batch_size": BATCH, "lag": LAG,
        "z_colloc_m": Z_COLLOC.tolist(), "z_sensor_m": Z_SENSOR,
        "dthdt_scale": DTHDT_SCALE,
    },
    "scalers": {
        "rain_max": RAIN_MAX,
        "vg_theta_r": VG["theta_r"],
        "vg_theta_s": VG["theta_s"],
        "scaler_y_min": scaler_y.data_min_.tolist(),
        "scaler_y_scale": scaler_y.scale_.tolist(),
        "T_MIN": T_MIN, "T_MAX": T_MAX, "Z_MAX": Z_MAX,
    },
    "vg_params": VG,
    "slope_params": dict(slope_deg=33.0, c_kPa=10.0, phi_deg=28.0, gamma_s=14.715, H_m=1.0),
    "device_id": 108,
    "train_losses": train_losses, "val_losses": val_losses,
    "data_losses": data_losses,   "phys_losses": phys_losses,
    "lr_history": lr_history,
    "best_val_loss": best_val,
    "timestamp": datetime.datetime.now().isoformat(),
}, model_path)
print(f"  → Model saved: {model_path}")

# ═══════════════════════════════════════════════════════════════
# ██  ABLATION — ML BASELINE (LAM=0, LAM_ZVAR=0, Data-Only)  ██
# ═══════════════════════════════════════════════════════════════
print("\n" + "█"*65)
print("  ABLATION: Training ML Baseline (Vanilla MLP, Data-Only)")
print("  LAM=0  LAM_ZVAR=0  — No physics constraint")
print("█"*65)

# ── Re-init same architecture from scratch (same seed) ──────
torch.manual_seed(42); np.random.seed(42)
model_ml = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
print(f"  ML Baseline Parameters: {sum(p.numel() for p in model_ml.parameters()):,}")

optimizer_ml = optim.AdamW(model_ml.parameters(), lr=LR, weight_decay=1e-4)
scheduler_ml = optim.lr_scheduler.LambdaLR(optimizer_ml, lr_lambda)

data_losses_ml  = []
train_losses_ml = []
val_losses_ml   = []

best_val_ml   = np.inf
best_state_ml = None

EPOCHS_ML = EPOCHS  # same number of epochs for fair comparison

for epoch in range(1, EPOCHS_ML + 1):
    model_ml.train()
    ep_data_ml = 0.0

    for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
        optimizer_ml.zero_grad()
        B = Xb.shape[0]

        # ── Data loss ONLY (no physics, no z-grad) ─────────────
        z_d = z_sens_norm.expand(B, -1)          # no requires_grad needed
        t_d = tb_norm.clone()                     # no requires_grad needed
        theta_pred_ml = model_ml(z_d, t_d, Xb)

        pred_norm_ml = (theta_pred_ml - y_min_t) * y_scl_t
        loss_ml = mse_loss(pred_norm_ml, yb)     # ONLY data loss

        loss_ml.backward()
        torch.nn.utils.clip_grad_norm_(model_ml.parameters(), 1.0)
        optimizer_ml.step()
        ep_data_ml += loss_ml.item()

    scheduler_ml.step()

    model_ml.eval()
    with torch.no_grad():
        vl_ml = mse_loss(
            (model_ml(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t,
            yv
        ).item()

    n_batches = len(loader)
    data_losses_ml.append(ep_data_ml / n_batches)
    train_losses_ml.append(ep_data_ml / n_batches)
    val_losses_ml.append(vl_ml)

    if vl_ml < best_val_ml:
        best_val_ml   = vl_ml
        best_state_ml = {k: v.clone() for k, v in model_ml.state_dict().items()}

    if epoch % 100 == 0:
        print(f"  [ML] Epoch {epoch:4d} | Data {data_losses_ml[-1]:.5f} | "
              f"Val {vl_ml:.5f} | LR {optimizer_ml.param_groups[0]['lr']:.2e}")

model_ml.load_state_dict(best_state_ml)
best_ep_ml = int(np.argmin(val_losses_ml)) + 1
print(f"\n  [ML] Best val: {best_val_ml:.5f}  @ epoch {best_ep_ml}")

def predict_sensor_ml(X_norm_np, t_sec_arr):
    model_ml.eval()
    N = X_norm_np.shape[0]
    z_s  = z_sens_norm.expand(N, -1)
    t_s  = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    Xn   = to_t(X_norm_np)
    with torch.no_grad():
        return model_ml(z_s, t_s, Xn).cpu().numpy().flatten()


# ═══════════════════════════════════════════════════════════════
# 7.  EVALUATE — PIML (Physics-Informed)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 7a] Evaluate PIML ...")
theta_pred_tr  = predict_sensor(X_tr,   t_sec_tr)
theta_pred_val = predict_sensor(X_val,  t_sec_val)
theta_pred_te  = predict_sensor(X_te,   t_sec_te)
theta_pred_all = predict_sensor(X_norm, t_sec_np)
theta_obs_all  = theta_np.copy()

r2_tr  = r2_score(theta_tr,  theta_pred_tr)
r2_val = r2_score(theta_val, theta_pred_val)
r2_te  = r2_score(theta_te,  theta_pred_te)
rmse_te = np.sqrt(mean_squared_error(theta_te, theta_pred_te))
residuals = theta_te - theta_pred_te

print(f"  [PIML] Train R²: {r2_tr:.4f}  |  Val R²: {r2_val:.4f}  |  Test R²: {r2_te:.4f}")
print(f"  [PIML] Test RMSE: {rmse_te:.5f} m³/m³")

rain_all = rain_np.copy()
ts       = pd.to_datetime(ts_all_np)

# ═══════════════════════════════════════════════════════════════
# 7b. EVALUATE — ML Baseline (Data-Only)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 7b] Evaluate ML Baseline ...")
theta_ml_tr  = predict_sensor_ml(X_tr,   t_sec_tr)
theta_ml_val = predict_sensor_ml(X_val,  t_sec_val)
theta_ml_te  = predict_sensor_ml(X_te,   t_sec_te)
theta_ml_all = predict_sensor_ml(X_norm, t_sec_np)

r2_ml_tr   = r2_score(theta_tr,  theta_ml_tr)
r2_ml_val  = r2_score(theta_val, theta_ml_val)
r2_ml_te   = r2_score(theta_te,  theta_ml_te)
rmse_ml_te = np.sqrt(mean_squared_error(theta_te, theta_ml_te))
residuals_ml = theta_te - theta_ml_te

print(f"  [ML]   Train R²: {r2_ml_tr:.4f}  |  Val R²: {r2_ml_val:.4f}  |  Test R²: {r2_ml_te:.4f}")
print(f"  [ML]   Test RMSE: {rmse_ml_te:.5f} m³/m³")


# ═══════════════════════════════════════════════════════════════
# 8.  Van Genuchten θ→ψ  — BOTH MODELS
# ═══════════════════════════════════════════════════════════════
print("\n[Step 8] Van Genuchten θ→ψ (both models) ...")

# ── PIML ──────────────────────────────────────────────────────
frac_low = (theta_pred_all < 0.10).mean()
psi_all_raw = vg_psi_np(theta_pred_all, VG)
psi_lo  = np.percentile(psi_all_raw, 2)
psi_all = np.clip(psi_all_raw, psi_lo, 0.0)
K_all   = vg_K_np(theta_pred_all, VG)
print(f"  [PIML] θ outlier frac (<0.10): {frac_low:.4f}")
print(f"  [PIML] |ψ| clipped range: [{-psi_all.max():.3f}, {-psi_all.min():.3f}] m")

# ── ML Baseline ───────────────────────────────────────────────
frac_low_ml = (theta_ml_all < 0.10).mean()
psi_ml_raw  = vg_psi_np(theta_ml_all, VG)
psi_lo_ml   = np.percentile(psi_ml_raw, 2)
psi_ml_all  = np.clip(psi_ml_raw, psi_lo_ml, 0.0)
K_ml_all    = vg_K_np(theta_ml_all, VG)
print(f"  [ML]   θ outlier frac (<0.10): {frac_low_ml:.4f}")
print(f"  [ML]   |ψ| clipped range: [{-psi_ml_all.max():.3f}, {-psi_ml_all.min():.3f}] m")

theta_vg = np.linspace(VG["theta_r"]+0.001, VG["theta_s"]-0.001, 300)
psi_vg   = vg_psi_np(theta_vg, VG)
kr_vg    = vg_Kr_np(theta_vg, VG)

# ═══════════════════════════════════════════════════════════════
# 9.  Pore pressure & FoS — BOTH MODELS
# ═══════════════════════════════════════════════════════════════
gamma_w = 9.81
SLOPE = 33.0; beta = np.radians(SLOPE)
c_ = 10.0; phi_ = np.radians(28.0)
gamma_s = 14.715; H = 1.0
sigma_n = gamma_s * H * np.cos(beta)**2
tau_d   = gamma_s * H * np.sin(beta) * np.cos(beta)

# PIML FoS
u_all   = gamma_w * psi_all
FoS_all = (c_ + np.maximum(sigma_n - u_all, 0) * np.tan(phi_)) / (tau_d + 1e-8)

# ML FoS
u_ml    = gamma_w * psi_ml_all
FoS_ml  = (c_ + np.maximum(sigma_n - u_ml, 0) * np.tan(phi_)) / (tau_d + 1e-8)

print(f"\n  [PIML] FoS min={FoS_all.min():.3f}  mean={FoS_all.mean():.3f}  FoS<1={(FoS_all<1).sum()}")
print(f"  [ML]   FoS min={FoS_ml.min():.3f}  mean={FoS_ml.mean():.3f}   FoS<1={(FoS_ml<1).sum()}")

# ═══════════════════════════════════════════════════════════════
# 10. Richards Residual — PIML (exact autograd) + ML (effective)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 10] Computing Richards residual ...")

# PIML residual via autograd
N_all  = X_norm.shape[0]
X_all_t = to_t(X_norm)
model.eval()
CHUNK = 1024
resid_list = []
for i in range(0, N_all, CHUNK):
    z_c = torch.full((min(CHUNK, N_all-i), 1), Z_SENSOR/Z_MAX,
                     dtype=torch.float32, device=DEVICE, requires_grad=True)
    t_c = to_t(((t_sec_np[i:i+CHUNK] - T_MIN)/(T_MAX-T_MIN))[:,None])
    t_c.requires_grad_(True)
    f_c = X_all_t[i:i+CHUNK]
    resid_c, _, _ = richards_residual(model, z_c, t_c, f_c)
    resid_list.append(resid_c.detach().cpu().numpy().flatten())

phys_residual = np.concatenate(resid_list)
mu_r = phys_residual.mean(); sd_r = phys_residual.std()
print(f"  [PIML] Richards residual  μ = {mu_r:.4e}  σ = {sd_r:.4e}  m³/m³/s")

# ML "effective" Richards residual (how much it violates physics)
resid_ml_list = []
model_ml.eval()
for i in range(0, N_all, CHUNK):
    z_c = torch.full((min(CHUNK, N_all-i), 1), Z_SENSOR/Z_MAX,
                     dtype=torch.float32, device=DEVICE, requires_grad=True)
    t_c = to_t(((t_sec_np[i:i+CHUNK] - T_MIN)/(T_MAX-T_MIN))[:,None])
    t_c.requires_grad_(True)
    f_c = X_all_t[i:i+CHUNK]
    resid_c, _, _ = richards_residual(model_ml, z_c, t_c, f_c)
    resid_ml_list.append(resid_c.detach().cpu().numpy().flatten())

phys_residual_ml = np.concatenate(resid_ml_list)
mu_r_ml = phys_residual_ml.mean(); sd_r_ml = phys_residual_ml.std()
print(f"  [ML]   Richards residual  μ = {mu_r_ml:.4e}  σ = {sd_r_ml:.4e}  m³/m³/s")


# ═══════════════════════════════════════════════════════════════
# ██████████  F I G U R E S  (Ablation Study)  ████████████████
# ═══════════════════════════════════════════════════════════════
print("\n[Step 11] Generating ablation study figures ...")

n_tr_plot  = len(theta_tr)
n_val_plot = n_tr_plot + len(theta_val)
epochs_arr = np.arange(1, EPOCHS + 1)
best_ep    = int(np.argmin(val_losses)) + 1

# colour shorthands
C_PIML = C["pred"]   # teal  — PIML
C_ML   = C["warn"]   # red   — ML baseline

# ──────────────────────────────────────────────────────────────
# FIG 1 — Loss Convergence Comparison (PIML vs ML)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(11, 12),
                         gridspec_kw={"hspace": 0.45})

ax = axes[0]
ax.semilogy(epochs_arr, train_losses,    color=C_PIML, lw=1.8, label="PIML train (data+physics)")
ax.semilogy(epochs_arr, val_losses,      color=C_PIML, lw=1.4, ls="--", alpha=0.8, label="PIML val")
ax.semilogy(epochs_arr, train_losses_ml, color=C_ML,   lw=1.8, label="ML train (data only)")
ax.semilogy(epochs_arr, val_losses_ml,   color=C_ML,   lw=1.4, ls="--", alpha=0.8, label="ML val")
ax.axvline(best_ep, color="gray", lw=1.0, ls=":", label=f"PIML best ep ({best_ep})")
ax.axvline(best_ep_ml, color="orange", lw=1.0, ls=":", label=f"ML best ep ({best_ep_ml})")
ax.set_ylabel("MSE Loss"); light_grid(ax)
ax.set_title("(A)  Train / Validation Loss — PIML vs ML Baseline", loc="left", fontweight="bold")
ax.legend(fontsize=8, ncol=2)

ax = axes[1]
ax.semilogy(epochs_arr, np.array(phys_losses), color=C["phys"], lw=1.8,
            label="PIML physics loss (normalised Richards PDE)")
ax.semilogy(epochs_arr, LAM * np.array(phys_losses), color=C["phys"], lw=1.2,
            ls="--", alpha=0.6, label=f"λ·Physics (λ={LAM})")
ax.axhline(1.0, color="gray", lw=0.8, ls=":", alpha=0.5, label="Reference = 1.0")
ax.set_ylabel("Richards PDE Loss (−)"); light_grid(ax)
ax.set_title("(B)  Physics Loss Convergence (PIML only)", loc="left", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[2]
data_arr = np.array(data_losses)
data_ml_arr = np.array(data_losses_ml)
ax.semilogy(epochs_arr, data_arr,    color=C_PIML, lw=1.8, label="PIML data component")
ax.semilogy(epochs_arr, data_ml_arr, color=C_ML,   lw=1.8, label="ML data loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Data MSE Loss"); light_grid(ax)
ax.set_title("(C)  Data Loss Component — PIML vs ML", loc="left", fontweight="bold")
ax.legend(fontsize=8)
ax.text(0.98, 0.05,
        f"PIML best val: {min(val_losses):.2e}\nML best val:   {min(val_losses_ml):.2e}",
        transform=ax.transAxes, fontsize=9, ha="right",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

fig.suptitle(
    f"Training Convergence — Ablation: PIML (λ={LAM}, λ_zvar={LAM_ZVAR}) vs Vanilla MLP (λ=0)\n"
    f"Same architecture · Same data · Same epochs ({EPOCHS}) · AdamW · Cosine LR",
    fontsize=11, fontweight="bold"
)
savefig(fig, "fig01_loss_convergence_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 2 — Scatter R² : PIML vs ML side by side (6 panels)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

splits_piml = [
    ("PIML — Train", theta_tr,  theta_pred_tr,  r2_tr,     C_PIML),
    ("PIML — Val",   theta_val, theta_pred_val, r2_val,    C_PIML),
    ("PIML — Test",  theta_te,  theta_pred_te,  r2_te,     C_PIML),
]
splits_ml = [
    ("ML — Train", theta_tr,  theta_ml_tr,  r2_ml_tr,  C_ML),
    ("ML — Val",   theta_val, theta_ml_val, r2_ml_val, C_ML),
    ("ML — Test",  theta_te,  theta_ml_te,  r2_ml_te,  C_ML),
]

for row, splits in enumerate([splits_piml, splits_ml]):
    for col, (label, obs, pred, r2, col_c) in enumerate(splits):
        ax = axes[row][col]
        ax.scatter(obs, pred, s=6, alpha=0.3, color=col_c, rasterized=True)
        lim = [min(obs.min(), pred.min())-0.002, max(obs.max(), pred.max())+0.002]
        ax.plot(lim, lim, "k--", lw=1.2, label="1:1")
        m, b = np.polyfit(obs, pred, 1)
        xfit = np.linspace(*lim, 100)
        ax.plot(xfit, m*xfit+b, color=col_c, lw=1.4, alpha=0.8, label="OLS")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("θ Observed (m³/m³)")
        ax.set_ylabel("θ Predicted (m³/m³)")
        rmse_s = np.sqrt(mean_squared_error(obs, pred))
        ax.set_title(f"{label}\nR²={r2:.4f}  RMSE={rmse_s:.4f}")
        ax.legend(fontsize=7); light_grid(ax)

fig.suptitle("Ablation Study — Scatter R²: PIML (top row) vs Vanilla MLP (bottom row)\n"
             "Device 108 · Full Richards Autograd · Same Architecture",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
savefig(fig, "fig02_scatter_r2_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 3 — θ Time-series Comparison (Test window)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios":[1, 2.5, 2.5], "hspace": 0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)"); light_grid(axes[0])
axes[0].set_title("A — Rainfall Intensity", fontweight="bold", loc="left")

axes[1].plot(ts, theta_obs_all,  color=C["obs"],  lw=1.3, alpha=0.9, label="θ Observed", zorder=3)
axes[1].plot(ts, theta_pred_all, color=C_PIML,    lw=1.3, alpha=0.9,
             label=f"PIML (R²={r2_te:.4f})", ls="--", zorder=2)
tr_end  = ts[n_tr_plot-1]; val_end = ts[n_val_plot-1]
axes[1].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"], label="Train")
axes[1].axvspan(tr_end, val_end, alpha=0.08, color=C["val"],   label="Val")
axes[1].axvspan(val_end, ts[-1], alpha=0.08, color=C["test"],  label="Test")
axes[1].set_ylabel("θ PIML (m³/m³)")
axes[1].legend(ncol=2, fontsize=8); light_grid(axes[1])
axes[1].set_title(f"B — PIML Prediction  [Test R²={r2_te:.4f}  RMSE={rmse_te:.4f}]",
                  fontweight="bold", loc="left")

axes[2].plot(ts, theta_obs_all, color=C["obs"], lw=1.3, alpha=0.9, label="θ Observed", zorder=3)
axes[2].plot(ts, theta_ml_all,  color=C_ML,     lw=1.3, alpha=0.9,
             label=f"ML Baseline (R²={r2_ml_te:.4f})", ls="--", zorder=2)
axes[2].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"])
axes[2].axvspan(tr_end, val_end, alpha=0.08, color=C["val"])
axes[2].axvspan(val_end, ts[-1], alpha=0.08, color=C["test"])
axes[2].set_ylabel("θ ML (m³/m³)"); axes[2].set_xlabel("Timestamp")
axes[2].legend(ncol=2, fontsize=8); light_grid(axes[2])
axes[2].set_title(f"C — ML Baseline Prediction  [Test R²={r2_ml_te:.4f}  RMSE={rmse_ml_te:.4f}]",
                  fontweight="bold", loc="left")

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle("θ Time-Series: PIML vs Vanilla MLP — Device 108", fontsize=12, fontweight="bold")
savefig(fig, "fig03_theta_timeseries_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 4 — Residual Comparison (Test set)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
ts_te_plot = ts[n_val_plot:]

# PIML residuals over time
axes[0][0].axhline(0, color="black", lw=0.8)
axes[0][0].fill_between(ts_te_plot, residuals, 0, where=(residuals>=0),
                        color=C_PIML, alpha=0.5, label="Over-pred")
axes[0][0].fill_between(ts_te_plot, residuals, 0, where=(residuals<0),
                        color=C["phys"], alpha=0.5, label="Under-pred")
axes[0][0].set_title("PIML Residuals vs Time (Test)"); axes[0][0].legend(fontsize=8)
axes[0][0].set_ylabel("θ_obs − θ_pred (m³/m³)"); light_grid(axes[0][0])
for t in axes[0][0].get_xticklabels(): t.set_rotation(15)

# PIML residual histogram
axes[0][1].hist(residuals, bins=40, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.85)
axes[0][1].axvline(0, color="black", lw=1.0, ls="--")
axes[0][1].axvline(residuals.mean(), color=C_ML, lw=1.4, ls="--",
                   label=f"μ={residuals.mean():.4f}")
axes[0][1].axvline(residuals.std(), color="gray", lw=1.2, ls=":",
                   label=f"σ={residuals.std():.4f}")
axes[0][1].axvline(-residuals.std(), color="gray", lw=1.2, ls=":")
axes[0][1].set_title("PIML Residual Distribution (Test)")
axes[0][1].set_xlabel("Residual (m³/m³)"); axes[0][1].legend(fontsize=8); light_grid(axes[0][1])

# ML residuals over time
axes[1][0].axhline(0, color="black", lw=0.8)
axes[1][0].fill_between(ts_te_plot, residuals_ml, 0, where=(residuals_ml>=0),
                        color=C_ML, alpha=0.5, label="Over-pred")
axes[1][0].fill_between(ts_te_plot, residuals_ml, 0, where=(residuals_ml<0),
                        color=C["u"], alpha=0.5, label="Under-pred")
axes[1][0].set_title("ML Baseline Residuals vs Time (Test)"); axes[1][0].legend(fontsize=8)
axes[1][0].set_ylabel("θ_obs − θ_pred (m³/m³)"); axes[1][0].set_xlabel("Timestamp")
light_grid(axes[1][0])
for t in axes[1][0].get_xticklabels(): t.set_rotation(15)

# ML residual histogram
axes[1][1].hist(residuals_ml, bins=40, color=C_ML, edgecolor="white", lw=0.3, alpha=0.85)
axes[1][1].axvline(0, color="black", lw=1.0, ls="--")
axes[1][1].axvline(residuals_ml.mean(), color=C_PIML, lw=1.4, ls="--",
                   label=f"μ={residuals_ml.mean():.4f}")
axes[1][1].axvline(residuals_ml.std(), color="gray", lw=1.2, ls=":",
                   label=f"σ={residuals_ml.std():.4f}")
axes[1][1].axvline(-residuals_ml.std(), color="gray", lw=1.2, ls=":")
axes[1][1].set_title("ML Baseline Residual Distribution (Test)")
axes[1][1].set_xlabel("Residual (m³/m³)"); axes[1][1].legend(fontsize=8); light_grid(axes[1][1])

fig.suptitle("Residual Analysis — PIML (top) vs Vanilla MLP (bottom) — Test Set  Dev108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig04_residuals_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 5 — Van Genuchten (same as original — PIML only)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(-psi_vg, theta_vg, color=C["vg1"], lw=2.0, label="VG Model")
psi_te_abs = np.abs(vg_psi_np(theta_pred_te, VG))
sort_idx   = np.argsort(psi_te_abs)
axes[0].scatter(psi_te_abs[sort_idx[::5]], theta_pred_te[sort_idx[::5]],
                s=6, alpha=0.3, color=C_PIML, rasterized=True, label="PIML θ̂ (test)")
psi_ml_te_abs = np.abs(vg_psi_np(theta_ml_te, VG))
sort_idx_ml   = np.argsort(psi_ml_te_abs)
axes[0].scatter(psi_ml_te_abs[sort_idx_ml[::5]], theta_ml_te[sort_idx_ml[::5]],
                s=6, alpha=0.2, color=C_ML, rasterized=True, label="ML θ̂ (test)")
axes[0].set_xlabel("|ψ| Matric Suction (m)"); axes[0].set_ylabel("θ (m³/m³)")
axes[0].set_title("(a) Soil Water Characteristic — PIML vs ML"); axes[0].set_xscale("log")
axes[0].legend(fontsize=8); light_grid(axes[0])
axes[0].text(0.98, 0.06,
             f"θ_r={VG['theta_r']} θ_s={VG['theta_s']}\nα={VG['alpha']} n={VG['n']}",
             transform=axes[0].transAxes, fontsize=8, ha="right",
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

axes[1].plot(theta_vg, kr_vg, color=C["psi"], lw=2.0)
axes[1].fill_between(theta_vg, kr_vg, alpha=0.15, color=C["psi"])
axes[1].set_xlabel("θ (m³/m³)"); axes[1].set_ylabel("K_r (−)")
axes[1].set_title("(b) Relative Hydraulic Conductivity (VG)"); light_grid(axes[1])
fig.suptitle("Van Genuchten Curves — Dev108", fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig05_vg_characteristic_curves.png")

# ──────────────────────────────────────────────────────────────
# FIG 6 — Matric Suction Comparison (PIML vs ML)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True,
                         gridspec_kw={"height_ratios":[1,2,2,2], "hspace":0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("Matric Suction: PIML vs ML — Dev10  [ψ clipped 2–99th pct]", loc="left")
light_grid(axes[0])

axes[1].plot(ts, theta_pred_all, color=C_PIML, lw=1.1, alpha=0.9, label="PIML θ̂")
axes[1].plot(ts, theta_ml_all,   color=C_ML,   lw=1.1, alpha=0.7, label="ML θ̂", ls="--")
axes[1].plot(ts, theta_obs_all,  color=C["obs"], lw=0.8, alpha=0.5, ls=":", label="Observed")
axes[1].set_ylabel("θ (m³/m³)"); axes[1].legend(fontsize=8, ncol=3); light_grid(axes[1])

axes[2].plot(ts, -psi_all,    color=C_PIML, lw=1.4, label="PIML |ψ|")
axes[2].fill_between(ts, -psi_all, alpha=0.10, color=C_PIML)
axes[2].set_ylabel("|ψ| PIML (m)"); axes[2].legend(fontsize=8); light_grid(axes[2])

axes[3].plot(ts, -psi_ml_all, color=C_ML,   lw=1.4, label="ML |ψ|")
axes[3].fill_between(ts, -psi_ml_all, alpha=0.10, color=C_ML)
axes[3].set_ylabel("|ψ| ML (m)"); axes[3].set_xlabel("Timestamp")
axes[3].legend(fontsize=8); light_grid(axes[3])

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig06_matric_suction_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 7 — FoS Comparison Time-series
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True,
                         gridspec_kw={"height_ratios":[1, 2, 2.5, 2.5], "hspace":0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("Factor of Safety: PIML vs ML Baseline — Dev108", loc="left")
light_grid(axes[0])

axes[1].plot(ts, u_all,  color=C_PIML, lw=1.3, label="PIML pore pressure u")
axes[1].plot(ts, u_ml,   color=C_ML,   lw=1.3, label="ML pore pressure u", ls="--", alpha=0.7)
axes[1].axhline(0, color="gray", lw=0.7, ls=":")
axes[1].set_ylabel("u (kPa)"); axes[1].legend(fontsize=8, ncol=2); light_grid(axes[1])

axes[2].plot(ts, FoS_all, color=C_PIML, lw=1.4, label=f"PIML FoS  (min={FoS_all.min():.3f})")
axes[2].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[2].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[2].fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0),
                     color=C["warn"], alpha=0.35, label=f"Failure (n={(FoS_all<1.0).sum()})")
axes[2].set_ylabel("FoS — PIML (−)"); axes[2].legend(ncol=2, fontsize=8); light_grid(axes[2])
axes[2].set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

axes[3].plot(ts, FoS_ml, color=C_ML, lw=1.4, label=f"ML FoS  (min={FoS_ml.min():.3f})")
axes[3].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[3].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[3].fill_between(ts, FoS_ml, 1.0, where=(FoS_ml<1.0),
                     color=C["warn"], alpha=0.35, label=f"Failure (n={(FoS_ml<1.0).sum()})")
axes[3].set_ylabel("FoS — ML (−)"); axes[3].set_xlabel("Timestamp")
axes[3].legend(ncol=2, fontsize=8); light_grid(axes[3])
axes[3].set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig07_fos_timeseries_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 8 — FoS Distribution Comparison
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

all_fos = np.concatenate([FoS_all, FoS_ml])
bins    = np.linspace(all_fos.min()-0.05, all_fos.max()+0.05, 60)

axes[0].hist(FoS_all, bins=bins, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.75,
             label=f"PIML  (μ={FoS_all.mean():.3f}, min={FoS_all.min():.3f})")
axes[0].hist(FoS_ml,  bins=bins, color=C_ML,   edgecolor="white", lw=0.3, alpha=0.55,
             label=f"ML    (μ={FoS_ml.mean():.3f}, min={FoS_ml.min():.3f})")
axes[0].axvline(1.0, color="black",       lw=2.0, ls="--", label="FoS=1.0 ⚠")
axes[0].axvline(1.3, color="darkorange",  lw=1.5, ls=":",  label="FoS=1.3 Warning")
axes[0].set_xlabel("FoS (−)"); axes[0].set_ylabel("Count")
axes[0].set_title("(a) FoS Histogram — PIML vs ML"); axes[0].legend(fontsize=8); light_grid(axes[0])

sorted_piml = np.sort(FoS_all); exc_piml = 1 - np.arange(1, len(sorted_piml)+1)/len(sorted_piml)
sorted_ml   = np.sort(FoS_ml);  exc_ml   = 1 - np.arange(1, len(sorted_ml)+1)/len(sorted_ml)
axes[1].semilogy(sorted_piml, exc_piml, color=C_PIML, lw=2.0, label="PIML")
axes[1].semilogy(sorted_ml,   exc_ml,   color=C_ML,   lw=2.0, label="ML Baseline", ls="--")
axes[1].axvline(1.0, color="black",      lw=1.5, ls="--", label="FoS=1.0")
axes[1].axvline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3")
axes[1].set_xlabel("FoS (−)"); axes[1].set_ylabel("Exceedance Prob.")
axes[1].set_title("(b) FoS Exceedance Probability"); axes[1].legend(fontsize=8); light_grid(axes[1])

fig.suptitle("Factor of Safety Distribution: PIML vs Vanilla MLP — Dev108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig08_fos_distribution_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 9 — Richards PDE Residual: PIML vs ML
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# PIML residual time-series
axes[0][0].plot(ts, phys_residual, color=C_PIML, lw=0.8, alpha=0.85)
axes[0][0].axhline(0, color="gray", lw=0.7, ls=":")
axes[0][0].fill_between(ts, phys_residual, 0,
                         where=(np.abs(phys_residual) > 2*sd_r),
                         color=C["warn"], alpha=0.4, label=f">2σ  σ={sd_r:.2e}")
axes[0][0].set_title("PIML — Richards Residual ∂θ/∂t − ∇·flux  (m³/m³/s)")
axes[0][0].set_ylabel("Residual (m³/m³/s)"); axes[0][0].legend(fontsize=8); light_grid(axes[0][0])
axes[0][0].text(0.99, 0.97, f"μ={mu_r:.2e}\nσ={sd_r:.2e}",
                transform=axes[0][0].transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))
for t in axes[0][0].get_xticklabels(): t.set_rotation(15)

# PIML residual histogram
axes[0][1].hist(phys_residual, bins=50, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.85)
axes[0][1].axvline(0, color="black", lw=1.0, ls="--")
axes[0][1].axvline( sd_r, color="gray", lw=1.2, ls=":", label=f"±σ={sd_r:.2e}")
axes[0][1].axvline(-sd_r, color="gray", lw=1.2, ls=":")
axes[0][1].set_title("PIML — PDE Residual Distribution")
axes[0][1].set_xlabel("Residual (m³/m³/s)"); axes[0][1].legend(fontsize=8); light_grid(axes[0][1])

# ML residual time-series
axes[1][0].plot(ts, phys_residual_ml, color=C_ML, lw=0.8, alpha=0.85)
axes[1][0].axhline(0, color="gray", lw=0.7, ls=":")
axes[1][0].fill_between(ts, phys_residual_ml, 0,
                         where=(np.abs(phys_residual_ml) > 2*sd_r_ml),
                         color=C["warn"], alpha=0.4, label=f">2σ  σ={sd_r_ml:.2e}")
axes[1][0].set_title("ML Baseline — Effective Richards Residual (m³/m³/s)")
axes[1][0].set_ylabel("Residual (m³/m³/s)"); axes[1][0].set_xlabel("Timestamp")
axes[1][0].legend(fontsize=8); light_grid(axes[1][0])
axes[1][0].text(0.99, 0.97, f"μ={mu_r_ml:.2e}\nσ={sd_r_ml:.2e}",
                transform=axes[1][0].transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))
for t in axes[1][0].get_xticklabels(): t.set_rotation(15)

# ML residual histogram
axes[1][1].hist(phys_residual_ml, bins=50, color=C_ML, edgecolor="white", lw=0.3, alpha=0.85)
axes[1][1].axvline(0, color="black", lw=1.0, ls="--")
axes[1][1].axvline( sd_r_ml, color="gray", lw=1.2, ls=":", label=f"±σ={sd_r_ml:.2e}")
axes[1][1].axvline(-sd_r_ml, color="gray", lw=1.2, ls=":")
axes[1][1].set_title("ML Baseline — PDE Residual Distribution")
axes[1][1].set_xlabel("Residual (m³/m³/s)"); axes[1][1].legend(fontsize=8); light_grid(axes[1][1])

fig.suptitle(
    "Richards PDE Residual: PIML (physics-constrained, top) vs ML Baseline (unconstrained, bottom)\n"
    "Smaller σ → better physical consistency  |  PIML should have σ ≪ ML",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
savefig(fig, "fig09_pde_residual_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 10 — Full Dashboard (PIML — same as original)
# ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 17))
gs  = gridspec.GridSpec(5, 1, figure=fig, hspace=0.50)
ax0 = fig.add_subplot(gs[0])
ax0.fill_between(ts, rain_all, color=C["rain"], alpha=0.75, step="mid")
ax0.set_ylabel("Rainfall (mm/min)")
ax0.set_title("A — Rainfall Intensity", fontweight="bold", loc="left")
light_grid(ax0); add_panel_label(ax0, "(A)", y=1.08)

ax1 = fig.add_subplot(gs[1])
ax1.plot(ts, theta_obs_all,  color=C["obs"],  lw=1.2, alpha=0.85, label="θ Observed")
ax1.plot(ts, theta_pred_all, color=C_PIML,    lw=1.2, alpha=0.85, label="θ PIML-Richards", ls="--")
ax1.plot(ts, theta_ml_all,   color=C_ML,      lw=1.0, alpha=0.65, label="θ ML-Baseline",  ls=":")
ax1.set_ylabel("θ (m³/m³)")
ax1.set_title(f"B — Soil Moisture  [PIML R²={r2_te:.3f}  ML R²={r2_ml_te:.3f}]",
              fontweight="bold", loc="left")
ax1.legend(ncol=3, fontsize=8); light_grid(ax1); add_panel_label(ax1, "(B)", y=1.08)

ax2 = fig.add_subplot(gs[2])
ax2.plot(ts, -psi_all,    color=C_PIML, lw=1.3, label="PIML |ψ|")
ax2.plot(ts, -psi_ml_all, color=C_ML,   lw=1.1, alpha=0.7, label="ML |ψ|", ls="--")
ax2.fill_between(ts, -psi_all, alpha=0.08, color=C_PIML)
ax2.set_ylabel("|ψ| clipped (m)")
ax2.set_title("C — Matric Suction (VG, clipped)", fontweight="bold", loc="left")
ax2.legend(fontsize=8); light_grid(ax2); add_panel_label(ax2, "(C)", y=1.08)

ax3 = fig.add_subplot(gs[3])
ax3.plot(ts, u_all, color=C_PIML, lw=1.3, label="PIML u")
ax3.plot(ts, u_ml,  color=C_ML,   lw=1.1, alpha=0.7, label="ML u", ls="--")
ax3.axhline(0, color="gray", lw=0.7, ls=":")
ax3.fill_between(ts, u_all, 0, where=(u_all>0), color=C["warn"], alpha=0.2, label="Positive u")
ax3.set_ylabel("u Pore Pressure (kPa)")
ax3.set_title("D — Pore Water Pressure", fontweight="bold", loc="left")
ax3.legend(fontsize=8, ncol=3); light_grid(ax3); add_panel_label(ax3, "(D)", y=1.08)

ax4 = fig.add_subplot(gs[4])
ax4.plot(ts, FoS_all, color=C_PIML, lw=1.4,
         label=f"PIML FoS (min={FoS_all.min():.3f})")
ax4.plot(ts, FoS_ml,  color=C_ML,   lw=1.2, alpha=0.75, ls="--",
         label=f"ML FoS   (min={FoS_ml.min():.3f})")
ax4.axhline(1.0, color=C["warn"],    lw=1.8, ls="--", label="FoS=1.0 ⚠ Failure")
ax4.axhline(1.3, color="darkorange", lw=1.2, ls=":",  label="FoS=1.3 Warning")
ax4.fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0), color=C["warn"], alpha=0.35)
ax4.set_ylabel("FoS (−)"); ax4.set_xlabel("Timestamp")
ax4.set_title("E — Factor of Safety (Infinite Slope)", fontweight="bold", loc="left")
ax4.legend(ncol=2, fontsize=8); light_grid(ax4); add_panel_label(ax4, "(E)", y=1.08)
ax4.set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

for ax in [ax0, ax1, ax2, ax3, ax4]:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle(
    "Full Dashboard — PIML (Richards, Autograd) vs ML Baseline (Data-Only)\n"
    f"Device 108  |  λ_phys={LAM}  λ_zvar={LAM_ZVAR}  |  PIML R²={r2_te:.4f}  ML R²={r2_ml_te:.4f}",
    fontsize=12, fontweight="bold", y=1.002)
savefig(fig, "fig10_full_dashboard_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 11 — ABLATION TABLE (publication-ready matplotlib table)
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis("off")

col_labels = ["Metric", "PIML\n(Full Richards)", "ML Baseline\n(Data-Only)", "Δ / Winner"]

def delta_str(piml_val, ml_val, higher_better=True):
    """Return improvement string."""
    diff = piml_val - ml_val
    pct  = abs(diff / (ml_val + 1e-12)) * 100
    if higher_better:
        winner = "✅ PIML" if diff > 0 else "❌ ML"
    else:
        winner = "✅ PIML" if diff < 0 else "❌ ML"
    return f"{winner}  ({abs(diff):.4f}, {pct:.1f}%)"

row_data = [
    ["Train R²",
     f"{r2_tr:.4f}", f"{r2_ml_tr:.4f}",
     delta_str(r2_tr, r2_ml_tr, higher_better=True)],

    ["Val R²",
     f"{r2_val:.4f}", f"{r2_ml_val:.4f}",
     delta_str(r2_val, r2_ml_val, higher_better=True)],

    ["Test R²",
     f"{r2_te:.4f}", f"{r2_ml_te:.4f}",
     delta_str(r2_te, r2_ml_te, higher_better=True)],

    ["Test RMSE (m³/m³)",
     f"{rmse_te:.5f}", f"{rmse_ml_te:.5f}",
     delta_str(rmse_te, rmse_ml_te, higher_better=False)],

    ["PDE Residual μ (m³/m³/s)",
     f"{mu_r:.3e}", f"{mu_r_ml:.3e}",
     delta_str(abs(mu_r), abs(mu_r_ml), higher_better=False)],

    ["PDE Residual σ (m³/m³/s)",
     f"{sd_r:.3e}", f"{sd_r_ml:.3e}",
     delta_str(sd_r, sd_r_ml, higher_better=False)],

    ["FoS Mean",
     f"{FoS_all.mean():.3f}", f"{FoS_ml.mean():.3f}",
     "—"],

    ["FoS Min",
     f"{FoS_all.min():.3f}", f"{FoS_ml.min():.3f}",
     delta_str(FoS_all.min(), FoS_ml.min(), higher_better=True)],

    ["FoS < 1.0 events",
     f"{(FoS_all<1.0).sum()}", f"{(FoS_ml<1.0).sum()}",
     "Lower = fewer false alarms"],

    ["FoS < 1.3 events",
     f"{(FoS_all<1.3).sum()}", f"{(FoS_ml<1.3).sum()}",
     "Lower = fewer warnings"],

    ["θ outlier frac (< 0.10)",
     f"{frac_low:.4f}", f"{frac_low_ml:.4f}",
     delta_str(frac_low, frac_low_ml, higher_better=False)],

    ["Physics constraint",
     "✅ Richards PDE (LAM=5)", "❌ None (LAM=0)", "PIML physically consistent"],

    ["Architecture",
     "RichardsPINN (same)", "RichardsPINN (same)", "Identical — fair comparison"],

    ["Training epochs",
     f"{EPOCHS}", f"{EPOCHS_ML}", "Identical"],
]

tbl = ax.table(
    cellText=row_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    colWidths=[0.28, 0.20, 0.20, 0.32],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 2.1)

# Header styling
for j in range(4):
    tbl[0, j].set_facecolor("#023E8A")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

# Row alternating + highlight key rows
highlight_rows = {2, 3, 5}   # Test R², RMSE, PDE σ — most important
for i in range(1, len(row_data)+1):
    for j in range(4):
        if i in highlight_rows:
            tbl[i, j].set_facecolor("#E3F2FD")
        elif i % 2 == 0:
            tbl[i, j].set_facecolor("#F8F9FA")
        else:
            tbl[i, j].set_facecolor("white")
        if j == 3:
            tbl[i, j].set_text_props(color="#023E8A" if "✅ PIML" in str(row_data[i-1][j]) else
                                      "#E63946"      if "❌ ML"   in str(row_data[i-1][j]) else
                                      "#333333")

ax.set_title(
    f"Table 1 — Ablation Study Results: PIML vs Vanilla MLP Baseline\n"
    f"Device 108 · Sandy Clay Loam · Slope 25° · Full Richards Equation (Autograd)\n"
    f"PIML: LAM={LAM}, LAM_ZVAR={LAM_ZVAR}   |   ML Baseline: LAM=0, LAM_ZVAR=0",
    fontsize=11, fontweight="bold", pad=18
)
plt.tight_layout()
savefig(fig, "fig11_ablation_table.png")


# ═══════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS: Ks (hydraulic conductivity) & Slope angle β
# ═══════════════════════════════════════════════════════════════
print("\n" + "█"*65)
print("  SENSITIVITY ANALYSIS")
print("  Perturbing Ks and slope angle β independently")
print("█"*65)

import itertools

# ── Perturbation ranges ────────────────────────────────────────
KS_SCALES  = np.array([0.60, 0.70, 0.80, 0.90, 1.00,
                        1.10, 1.20, 1.30, 1.40])          # ×baseline
SLOPE_DEGS = np.array([23.0, 25.0, 27.0, 29.0, 31.0,
                        33.0, 35.0, 37.0, 39.0, 41.0])   # degrees

BASE_KS    = VG["Ks_ms"]      # baseline hydraulic conductivity (m/s)
BASE_SLOPE = 33.0             # baseline slope (degrees)

# ── Fixed geotechnical params ──────────────────────────────────
c_kPa   = 10.0
phi_deg = 28.0
phi_rad = np.radians(phi_deg)
gamma_s = 14.715
H_m     = 1.0


def compute_fos_series(theta_arr, slope_deg, ks_scale=1.0):
    """
    Given θ predictions, slope angle, and Ks scale factor,
    compute ψ (VG, using scaled Ks), u, and FoS array.

    Note: ψ depends only on θ via VG — Ks doesn't change ψ directly.
    The Ks perturbation propagates through the flux term in the PDE,
    which we approximate here by noting that at equilibrium a higher Ks
    drains faster → lower θ at steady state. For the sensitivity study
    we propagate Ks uncertainty through the pore-pressure path only.
    """
    # ψ from VG (theta only — Ks doesn't enter ψ directly)
    psi_raw = vg_psi_np(theta_arr, VG)
    psi_lo  = np.percentile(psi_raw, 2)
    psi     = np.clip(psi_raw, psi_lo, 0.0)

    # Scale ψ linearly with Ks: higher Ks → faster drainage → less suction
    # Simple sensitivity proxy: ψ_eff = ψ / ks_scale
    # (conservative: more Ks → easier drainage → less negative ψ → higher u)
    psi_eff = psi / ks_scale

    beta_r  = np.radians(slope_deg)
    sigma_n = gamma_s * H_m * np.cos(beta_r)**2
    tau_d   = gamma_s * H_m * np.sin(beta_r) * np.cos(beta_r)

    u   = gamma_w * psi_eff

    u_clamped = np.clip(u, -sigma_n, None)
    effective_stress = np.maximum(sigma_n - u_clamped, 0)

    FoS = (c_kPa + effective_stress * np.tan(phi_rad)) / (tau_d + 1e-8)
    return FoS, u, psi_eff


# ══════════════════════════════════════════════════════════════
# (A) Ks sensitivity — vary Ks scale, keep slope = 33°
# ══════════════════════════════════════════════════════════════
print("\n[SA-A] Ks sensitivity ...")
ks_results = {}
for scale in KS_SCALES:
    fos, u_, psi_ = compute_fos_series(theta_pred_all, BASE_SLOPE, ks_scale=scale)
    ks_results[scale] = {
        "FoS_mean": fos.mean(), "FoS_min": fos.min(),
        "FoS_p5": np.percentile(fos, 5),
        "n_fail": (fos < 1.0).sum(),
        "n_warn": (fos < 1.3).sum(),
        "FoS_series": fos,
    }
    print(f"  Ks×{scale:.2f}  FoS_mean={fos.mean():.3f}  "
          f"FoS_min={fos.min():.3f}  fail={(fos<1.0).sum()}")

# ══════════════════════════════════════════════════════════════
# (B) Slope sensitivity — vary β, keep Ks = baseline
# ══════════════════════════════════════════════════════════════
print("\n[SA-B] Slope angle sensitivity ...")
slope_results = {}
for deg in SLOPE_DEGS:
    fos, u_, psi_ = compute_fos_series(theta_pred_all, deg, ks_scale=1.0)
    slope_results[deg] = {
        "FoS_mean": fos.mean(), "FoS_min": fos.min(),
        "FoS_p5": np.percentile(fos, 5),
        "n_fail": (fos < 1.0).sum(),
        "n_warn": (fos < 1.3).sum(),
        "FoS_series": fos,
    }
    print(f"  β={deg:.0f}°  FoS_mean={fos.mean():.3f}  "
          f"FoS_min={fos.min():.3f}  fail={(fos<1.0).sum()}")

# ══════════════════════════════════════════════════════════════
# (C) 2-D grid: Ks × Slope → FoS_min  (heatmap)
# ══════════════════════════════════════════════════════════════
print("\n[SA-C] 2-D sensitivity grid (Ks × slope) ...")
grid_fos_min  = np.zeros((len(SLOPE_DEGS), len(KS_SCALES)))
grid_fos_mean = np.zeros_like(grid_fos_min)
grid_n_fail   = np.zeros_like(grid_fos_min)

for i, deg in enumerate(SLOPE_DEGS):
    for j, scale in enumerate(KS_SCALES):
        fos, _, _ = compute_fos_series(theta_pred_all, deg, ks_scale=scale)
        grid_fos_min[i, j]  = fos.min()
        grid_fos_mean[i, j] = fos.mean()
        grid_n_fail[i, j]   = (fos < 1.0).sum()

print("  Grid complete.")


# ═══════════════════════════════════════════════════════════════
# COMPUTATIONAL EFFICIENCY — Inference Speed Benchmark
# PINN vs HYDRUS-1D proxy vs Real-time EWS requirement
# ═══════════════════════════════════════════════════════════════
print("\n" + "█"*65)
print("  COMPUTATIONAL EFFICIENCY ANALYSIS")
print("  PINN inference speed · HYDRUS comparison · EWS feasibility")
print("█"*65)

import timeit
import platform

# ── Hardware info ──────────────────────────────────────────────
print(f"\n  Platform : {platform.system()} {platform.machine()}")
print(f"  PyTorch  : {torch.__version__}")
print(f"  Device   : {DEVICE}")

# ── Benchmark settings ─────────────────────────────────────────
N_WARMUP   = 20     # warmup runs (discard)
N_REPEATS  = 200    # timed runs
BATCH_SIZES = [1, 8, 32, 64, 128, 256, 512]

# HYDRUS-1D literature reference runtimes
# Source: Šimůnek et al. (2009); typical 1D transient run
# for a 1 m profile, 24 h simulation, fine grid ≈ 30–120 s on a desktop CPU
HYDRUS_TIME_SEC  = 60.0   # conservative midpoint estimate (seconds)
HYDRUS_LABEL     = "HYDRUS-1D\n(lit. est. ~60 s)"

# EWS real-time requirement
EWS_LIMIT_SEC = 60.0   # end-to-end pipeline must complete within 60 s

# ── Single-sample inference benchmark ─────────────────────────
print("\n[CE-1] Single-sample (batch=1) latency benchmark ...")

model.eval()

def single_inference():
    """One forward pass: 1 sample, sensor depth, current t."""
    with torch.no_grad():
        z_s = z_sens_norm                                    # (1,1)
        t_s = to_t(np.array([[t_sec_np[-1]]], dtype=np.float32))
        t_s = (t_s - T_MIN) / (T_MAX - T_MIN)
        x_s = to_t(X_norm[-1:])
        _ = model(z_s, t_s, x_s)

# Warmup
for _ in range(N_WARMUP):
    single_inference()

# Timed runs
times_single = []
for _ in range(N_REPEATS):
    t0 = timeit.default_timer()
    single_inference()
    times_single.append(timeit.default_timer() - t0)

times_single = np.array(times_single) * 1000  # → ms
t_mean = times_single.mean()
t_std  = times_single.std()
t_p50  = np.percentile(times_single, 50)
t_p95  = np.percentile(times_single, 95)
t_p99  = np.percentile(times_single, 99)

print(f"  Single inference  mean={t_mean:.3f} ms  std={t_std:.3f} ms")
print(f"  p50={t_p50:.3f}  p95={t_p95:.3f}  p99={t_p99:.3f} ms")
print(f"  Throughput: {1000/t_mean:.1f} samples/s")

# ── Batch inference benchmark (multiple batch sizes) ───────────
print("\n[CE-2] Batch inference benchmark ...")

batch_results = {}
for bs in BATCH_SIZES:
    idx = np.random.choice(len(X_norm), bs, replace=(bs > len(X_norm)))

    def batch_inference(bs=bs, idx=idx):
        with torch.no_grad():
            z_b = z_sens_norm.expand(bs, -1)
            t_b = to_t(((t_sec_np[idx] - T_MIN) / (T_MAX - T_MIN))[:, None])
            x_b = to_t(X_norm[idx])
            _ = model(z_b, t_b, x_b)

    # Warmup
    for _ in range(N_WARMUP):
        batch_inference()

    times_b = []
    for _ in range(N_REPEATS):
        t0 = timeit.default_timer()
        batch_inference()
        times_b.append(timeit.default_timer() - t0)

    times_b = np.array(times_b) * 1000
    batch_results[bs] = {
        "mean_ms"      : times_b.mean(),
        "std_ms"       : times_b.std(),
        "per_sample_ms": times_b.mean() / bs,
        "throughput"   : bs / (times_b.mean() / 1000),
    }
    print(f"  Batch={bs:4d} | total={times_b.mean():.3f} ms "
          f"| per-sample={times_b.mean()/bs:.4f} ms "
          f"| throughput={bs/(times_b.mean()/1000):.0f} /s")

# ── Full pipeline timing (θ → ψ → u → FoS for all test data) ──
print("\n[CE-3] Full pipeline timing (all test samples) ...")

N_te = len(X_te)

def full_pipeline():
    # Step 1: PINN inference
    theta_out = predict_sensor(X_te, t_sec_te)
    # Step 2: VG → ψ
    psi_out   = vg_psi_np(theta_out, VG)
    psi_out   = np.clip(psi_out, np.percentile(psi_out, 2), 0.0)
    # Step 3: u → FoS
    beta_r    = np.radians(BASE_SLOPE)
    sigma_n   = gamma_s * H_m * np.cos(beta_r)**2
    tau_d     = gamma_s * H_m * np.sin(beta_r) * np.cos(beta_r)
    u_out     = gamma_w * psi_out
    fos_out   = (c_kPa + np.maximum(sigma_n - u_out, 0) * np.tan(phi_rad)) / (tau_d + 1e-8)
    return fos_out

# Warmup
for _ in range(5):
    full_pipeline()

times_pipe = []
for _ in range(50):
    t0 = timeit.default_timer()
    full_pipeline()
    times_pipe.append(timeit.default_timer() - t0)

times_pipe   = np.array(times_pipe) * 1000
pipe_mean_ms = times_pipe.mean()
pipe_std_ms  = times_pipe.std()
pipe_sec     = pipe_mean_ms / 1000.0

print(f"  Full pipeline ({N_te} samples): {pipe_mean_ms:.2f} ± {pipe_std_ms:.2f} ms")
print(f"  = {pipe_sec:.4f} s  (EWS limit: {EWS_LIMIT_SEC} s)")
print(f"  EWS feasible: {'✅ YES' if pipe_sec < EWS_LIMIT_SEC else '❌ NO'}")

# ── Speedup vs HYDRUS ──────────────────────────────────────────
speedup_single = HYDRUS_TIME_SEC / (t_mean / 1000)
speedup_pipe   = HYDRUS_TIME_SEC / pipe_sec
pct_faster     = (1 - pipe_sec / HYDRUS_TIME_SEC) * 100

print(f"\n  Speedup (single inference vs HYDRUS): {speedup_single:.0f}×")
print(f"  Speedup (full pipeline   vs HYDRUS): {speedup_pipe:.0f}×")
print(f"  PINN pipeline is {pct_faster:.1f}% faster than HYDRUS-1D")

# ── ML baseline comparison ─────────────────────────────────────
print("\n[CE-4] ML baseline inference speed ...")

def single_inference_ml():
    with torch.no_grad():
        z_s = z_sens_norm
        t_s = to_t(np.array([[t_sec_np[-1]]], dtype=np.float32))
        t_s = (t_s - T_MIN) / (T_MAX - T_MIN)
        x_s = to_t(X_norm[-1:])
        _ = model_ml(z_s, t_s, x_s)

for _ in range(N_WARMUP):
    single_inference_ml()

times_ml = []
for _ in range(N_REPEATS):
    t0 = timeit.default_timer()
    single_inference_ml()
    times_ml.append(timeit.default_timer() - t0)

times_ml   = np.array(times_ml) * 1000
t_mean_ml  = times_ml.mean()
print(f"  ML baseline single: {t_mean_ml:.3f} ms  "
      f"(PIML: {t_mean:.3f} ms, overhead: {t_mean - t_mean_ml:+.3f} ms)")



# ──────────────────────────────────────────────────────────────
# FINAL RESULTS SUMMARY (both models)
# ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  ABLATION STUDY — FINAL RESULTS (Dev108)")
print("  PIML: Full 1D Richards  vs  ML: Vanilla MLP (Data-Only)")
print("="*70)
print(f"  {'Metric':<30} {'PIML':>12} {'ML Baseline':>14}")
print(f"  {'-'*58}")
print(f"  {'Train R²':<30} {r2_tr:>12.4f} {r2_ml_tr:>14.4f}")
print(f"  {'Val   R²':<30} {r2_val:>12.4f} {r2_ml_val:>14.4f}")
print(f"  {'Test  R²':<30} {r2_te:>12.4f} {r2_ml_te:>14.4f}")
print(f"  {'Test  RMSE (m³/m³)':<30} {rmse_te:>12.5f} {rmse_ml_te:>14.5f}")
print(f"  {'PDE Residual μ':<30} {mu_r:>12.4e} {mu_r_ml:>14.4e}")
print(f"  {'PDE Residual σ':<30} {sd_r:>12.4e} {sd_r_ml:>14.4e}")
print(f"  {'FoS Mean':<30} {FoS_all.mean():>12.3f} {FoS_ml.mean():>14.3f}")
print(f"  {'FoS Min':<30} {FoS_all.min():>12.3f} {FoS_ml.min():>14.3f}")
print(f"  {'FoS < 1.0 events':<30} {(FoS_all<1.0).sum():>12d} {(FoS_ml<1.0).sum():>14d}")
print(f"  {'FoS < 1.3 events':<30} {(FoS_all<1.3).sum():>12d} {(FoS_ml<1.3).sum():>14d}")
print(f"  {'θ outlier frac':<30} {frac_low:>12.4f} {frac_low_ml:>14.4f}")
print("="*70)

piml_wins = sum([
    r2_te > r2_ml_te,
    rmse_te < rmse_ml_te,
    abs(sd_r) < abs(sd_r_ml),
    FoS_all.min() > FoS_ml.min(),
])
print(f"\n  PIML wins on {piml_wins}/4 key metrics (R², RMSE, PDE σ, FoS min)")
print(f"\n✅ Figures → {OUT}/")
print(f"   fig01 — Loss convergence ablation")
print(f"   fig02 — Scatter R² ablation (6 panels)")
print(f"   fig03 — θ time-series ablation")
print(f"   fig04 — Residuals ablation")
print(f"   fig05 — VG characteristic curves")
print(f"   fig06 — Matric suction ablation")
print(f"   fig07 — FoS time-series ablation")
print(f"   fig08 — FoS distribution ablation")
print(f"   fig09 — PDE residual ablation")
print(f"   fig10 — Full dashboard ablation")
print(f"   fig11 — Ablation TABLE (publication-ready)")


# ══════════════════════════════════════════════════════════════
# SENSITIVITY FIGURES
# ══════════════════════════════════════════════════════════════
print("\n[SA Figures] Generating sensitivity analysis figures ...")


# ── FIG SA-1 — Ks sensitivity: FoS band plot ──────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                         gridspec_kw={"height_ratios": [1, 2.5], "hspace": 0.15},
                         sharex=True)

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("SA-1 — Ks Sensitivity: FoS Band  (slope fixed at 33°)", loc="left", fontweight="bold")
light_grid(axes[0])

cmap_ks = plt.cm.coolwarm
norm_ks = plt.Normalize(KS_SCALES.min(), KS_SCALES.max())

for scale in KS_SCALES:
    fos_s = ks_results[scale]["FoS_series"]
    col   = cmap_ks(norm_ks(scale))
    lw    = 2.0 if np.isclose(scale, 1.0) else 0.9
    ls    = "-" if np.isclose(scale, 1.0) else "--"
    label = f"Ks×{scale:.2f}" if (np.isclose(scale, 0.60) or
                                    np.isclose(scale, 1.00) or
                                    np.isclose(scale, 1.40)) else None
    axes[1].plot(ts, fos_s, color=col, lw=lw, ls=ls, alpha=0.8, label=label)

# Uncertainty band (min–max across all Ks scales)
fos_stack = np.vstack([ks_results[s]["FoS_series"] for s in KS_SCALES])
axes[1].fill_between(ts, fos_stack.min(axis=0), fos_stack.max(axis=0),
                     alpha=0.10, color="gray", label="Ks uncertainty band")
axes[1].axhline(1.0, color=C["warn"],    lw=1.8, ls="--", label="FoS=1.0 ⚠")
axes[1].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[1].set_ylabel("FoS (−)"); axes[1].set_xlabel("Timestamp")
axes[1].legend(ncol=3, fontsize=8); light_grid(axes[1])
axes[1].set_xlim(ts[0], ts[-1])
for t in axes[1].get_xticklabels(): t.set_rotation(15)

sm = plt.cm.ScalarMappable(cmap=cmap_ks, norm=norm_ks)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], pad=0.01, fraction=0.02)
cbar.set_label("Ks scale factor", fontsize=9)

fig.suptitle("Sensitivity Analysis — Hydraulic Conductivity (Ks) — Device 108",
             fontsize=12, fontweight="bold")
savefig(fig, "figSA1_ks_sensitivity_band.png")


# ── FIG SA-2 — Slope sensitivity: FoS band plot ───────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                         gridspec_kw={"height_ratios": [1, 2.5], "hspace": 0.15},
                         sharex=True)

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm/min)")
axes[0].set_title("SA-2 — Slope Angle Sensitivity: FoS Band  (Ks fixed at baseline)", loc="left", fontweight="bold")
light_grid(axes[0])

cmap_sl = plt.cm.RdYlGn_r
norm_sl = plt.Normalize(SLOPE_DEGS.min(), SLOPE_DEGS.max())

for deg in SLOPE_DEGS:
    fos_s = slope_results[deg]["FoS_series"]
    col   = cmap_sl(norm_sl(deg))
    lw    = 2.0 if np.isclose(deg, BASE_SLOPE) else 0.9
    ls    = "-" if np.isclose(deg, BASE_SLOPE) else "--"
    label = f"β={deg:.0f}°" if (np.isclose(deg, SLOPE_DEGS[0]) or
                                  np.isclose(deg, BASE_SLOPE) or
                                  np.isclose(deg, SLOPE_DEGS[-1])) else None
    axes[1].plot(ts, fos_s, color=col, lw=lw, ls=ls, alpha=0.8, label=label)

fos_stack_sl = np.vstack([slope_results[d]["FoS_series"] for d in SLOPE_DEGS])
axes[1].fill_between(ts, fos_stack_sl.min(axis=0), fos_stack_sl.max(axis=0),
                     alpha=0.10, color="gray", label="Slope uncertainty band")
axes[1].axhline(1.0, color=C["warn"],    lw=1.8, ls="--", label="FoS=1.0 ⚠")
axes[1].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[1].set_ylabel("FoS (−)"); axes[1].set_xlabel("Timestamp")
axes[1].legend(ncol=3, fontsize=8); light_grid(axes[1])
axes[1].set_xlim(ts[0], ts[-1])
for t in axes[1].get_xticklabels(): t.set_rotation(15)

sm2 = plt.cm.ScalarMappable(cmap=cmap_sl, norm=norm_sl)
sm2.set_array([])
cbar2 = fig.colorbar(sm2, ax=axes[1], pad=0.01, fraction=0.02)
cbar2.set_label("Slope angle (°)", fontsize=9)

fig.suptitle("Sensitivity Analysis — Slope Angle (β) — Device 108",
             fontsize=12, fontweight="bold")
savefig(fig, "figSA2_slope_sensitivity_band.png")


# ── FIG SA-3 — Tornado chart (±20% perturbation) ──────────────
fig, ax = plt.subplots(figsize=(10, 5))

# Baseline FoS_mean
baseline_fos_mean = ks_results[1.0]["FoS_mean"]

# Ks ±20%: find closest scales
ks_lo = ks_results[min(KS_SCALES, key=lambda s: abs(s - 0.80))]["FoS_mean"]
ks_hi = ks_results[min(KS_SCALES, key=lambda s: abs(s - 1.20))]["FoS_mean"]

# Slope ±20% in absolute degrees (33 ± ~6°)
sl_lo = slope_results[min(SLOPE_DEGS, key=lambda d: abs(d - 27.0))]["FoS_mean"]
sl_hi = slope_results[min(SLOPE_DEGS, key=lambda d: abs(d - 39.0))]["FoS_mean"]

params  = ["Hydraulic conductivity (Ks)", "Slope angle (β)"]
lo_vals = [ks_lo - baseline_fos_mean, sl_lo - baseline_fos_mean]
hi_vals = [ks_hi - baseline_fos_mean, sl_hi - baseline_fos_mean]

y_pos = np.arange(len(params))
bars_lo = ax.barh(y_pos, lo_vals, left=baseline_fos_mean,
                  color=C["phys"], alpha=0.8, label="−20% perturbation", height=0.4)
bars_hi = ax.barh(y_pos, hi_vals, left=baseline_fos_mean,
                  color=C["pred"], alpha=0.8, label="+20% perturbation", height=0.4)
ax.axvline(baseline_fos_mean, color="black", lw=1.5, ls="--",
           label=f"Baseline FoS_mean = {baseline_fos_mean:.3f}")
ax.axvline(1.0, color=C["warn"], lw=1.2, ls=":", label="FoS=1.0 ⚠")
ax.set_yticks(y_pos); ax.set_yticklabels(params, fontsize=11)
ax.set_xlabel("FoS Mean (−)"); light_grid(ax, axis="x")
ax.legend(fontsize=9)
ax.set_title("Tornado Chart — Parameter Sensitivity on Mean FoS  (±20% perturbation)\nDevice 108",
             fontweight="bold")

# Annotate values
for i, (lo, hi) in enumerate(zip(lo_vals, hi_vals)):
    ax.text(baseline_fos_mean + lo - 0.005, i, f"{lo:+.3f}", ha="right",
            va="center", fontsize=9, color="white", fontweight="bold")
    ax.text(baseline_fos_mean + hi + 0.005, i, f"{hi:+.3f}", ha="left",
            va="center", fontsize=9, color="white", fontweight="bold")

plt.tight_layout()
savefig(fig, "figSA3_tornado_chart.png")


# ── FIG SA-4 — 2-D Heatmap: FoS_min (Ks × slope) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ks_labels    = [f"×{s:.2f}" for s in KS_SCALES]
slope_labels = [f"{d:.0f}°" for d in SLOPE_DEGS]

im1 = axes[0].imshow(grid_fos_min, aspect="auto", origin="lower",
                      cmap="RdYlGn", vmin=0.8, vmax=2.5)
axes[0].set_xticks(range(len(KS_SCALES)));    axes[0].set_xticklabels(ks_labels,    fontsize=8, rotation=45)
axes[0].set_yticks(range(len(SLOPE_DEGS)));   axes[0].set_yticklabels(slope_labels, fontsize=8)
axes[0].set_xlabel("Ks scale factor"); axes[0].set_ylabel("Slope angle (°)")
axes[0].set_title("(a) FoS Min  — Ks × Slope", fontweight="bold")
cbar1 = fig.colorbar(im1, ax=axes[0], pad=0.02)
cbar1.set_label("FoS Min (−)")
# Annotate cells
for i in range(len(SLOPE_DEGS)):
    for j in range(len(KS_SCALES)):
        val = grid_fos_min[i, j]
        col = "white" if val < 1.3 else "black"
        axes[0].text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=7, color=col, fontweight="bold")
# Mark failure zone
axes[0].contour(grid_fos_min, levels=[1.0], colors="red",   linewidths=2)
axes[0].contour(grid_fos_min, levels=[1.3], colors="orange", linewidths=1.5, linestyles="--")

im2 = axes[1].imshow(grid_n_fail, aspect="auto", origin="lower",
                      cmap="YlOrRd")
axes[1].set_xticks(range(len(KS_SCALES)));    axes[1].set_xticklabels(ks_labels,    fontsize=8, rotation=45)
axes[1].set_yticks(range(len(SLOPE_DEGS)));   axes[1].set_yticklabels(slope_labels, fontsize=8)
axes[1].set_xlabel("Ks scale factor"); axes[1].set_ylabel("Slope angle (°)")
axes[1].set_title("(b) FoS<1.0 event count — Ks × Slope", fontweight="bold")
cbar2 = fig.colorbar(im2, ax=axes[1], pad=0.02)
cbar2.set_label("Count (timesteps)")
for i in range(len(SLOPE_DEGS)):
    for j in range(len(KS_SCALES)):
        val = grid_n_fail[i, j]
        col = "white" if val > grid_n_fail.max()*0.5 else "black"
        axes[1].text(j, i, f"{int(val)}", ha="center", va="center",
                     fontsize=7, color=col, fontweight="bold")

fig.suptitle("2-D Sensitivity Heatmap: Ks × Slope Angle — Device 108\n"
             "Red contour = FoS<1.0 boundary  |  Orange dashed = FoS<1.3",
             fontsize=11, fontweight="bold")
plt.tight_layout()
savefig(fig, "figSA4_2d_sensitivity_heatmap.png")


# ── FIG SA-5 — Summary: FoS metrics vs Ks and vs slope ────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)

metrics = ["FoS_mean", "FoS_min", "FoS_p5", "n_fail", "n_warn"]
titles  = ["FoS Mean", "FoS Min", "FoS 5th percentile", "Count FoS<1.0", "Count FoS<1.3"]

for col_i, (met, ttl) in enumerate(zip(["FoS_mean","FoS_min","FoS_p5"], titles[:3])):
    ax = axes[0][col_i]
    vals = [ks_results[s][met] for s in KS_SCALES]
    ax.plot(KS_SCALES, vals, "o-", color=C["phys"], lw=1.8, ms=6)
    ax.axhline(1.0, color=C["warn"],    lw=1.0, ls="--", alpha=0.7)
    ax.axhline(1.3, color="darkorange", lw=0.8, ls=":",  alpha=0.7)
    ax.axvline(1.0, color="gray", lw=0.8, ls=":")
    ax.set_xlabel("Ks scale factor"); ax.set_ylabel(ttl); ax.set_title(f"Ks → {ttl}", fontweight="bold")
    light_grid(ax)

for col_i, (met, ttl) in enumerate(zip(["FoS_mean","FoS_min","FoS_p5"], titles[:3])):
    ax = axes[1][col_i]
    vals = [slope_results[d][met] for d in SLOPE_DEGS]
    ax.plot(SLOPE_DEGS, vals, "s-", color=C["coral"], lw=1.8, ms=6)
    ax.axhline(1.0, color=C["warn"],    lw=1.0, ls="--", alpha=0.7)
    ax.axhline(1.3, color="darkorange", lw=0.8, ls=":",  alpha=0.7)
    ax.axvline(BASE_SLOPE, color="gray", lw=0.8, ls=":")
    ax.set_xlabel("Slope angle (°)"); ax.set_ylabel(ttl); ax.set_title(f"Slope → {ttl}", fontweight="bold")
    light_grid(ax)

fig.suptitle("Sensitivity Metrics vs Ks (top row) and Slope Angle (bottom row) — Device 108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "figSA5_sensitivity_metrics_curves.png")

# ── Print summary ──────────────────────────────────────────────
print("\n" + "="*65)
print("  SENSITIVITY ANALYSIS SUMMARY — Device 108")
print("="*65)
print(f"\n  Ks sensitivity (β=33° fixed):")
print(f"  {'Ks scale':<12} {'FoS mean':>10} {'FoS min':>10} {'Fail':>8}")
for s in KS_SCALES:
    r = ks_results[s]
    print(f"  {s:<12.2f} {r['FoS_mean']:>10.3f} {r['FoS_min']:>10.3f} {r['n_fail']:>8d}")

print(f"\n  Slope sensitivity (Ks=baseline fixed):")
print(f"  {'Slope (°)':<12} {'FoS mean':>10} {'FoS min':>10} {'Fail':>8}")
for d in SLOPE_DEGS:
    r = slope_results[d]
    print(f"  {d:<12.1f} {r['FoS_mean']:>10.3f} {r['FoS_min']:>10.3f} {r['n_fail']:>8d}")

print(f"\n✅ Sensitivity figures → {OUT}/")
print("   figSA1 — Ks sensitivity band plot")
print("   figSA2 — Slope sensitivity band plot")
print("   figSA3 — Tornado chart")
print("   figSA4 — 2-D heatmap (Ks × slope)")
print("   figSA5 — Metric curves")


# ══════════════════════════════════════════════════════════════
# COMPUTATIONAL EFFICIENCY FIGURES
# ══════════════════════════════════════════════════════════════
print("\n[CE Figures] Generating efficiency figures ...")

# ── FIG CE-1 — Inference latency distribution ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(times_single, bins=40, color=C["pred"],
             edgecolor="white", lw=0.3, alpha=0.85)
axes[0].axvline(t_mean, color=C["warn"],  lw=2.0, ls="--",
                label=f"Mean = {t_mean:.3f} ms")
axes[0].axvline(t_p95,  color="orange",   lw=1.4, ls=":",
                label=f"p95 = {t_p95:.3f} ms")
axes[0].axvline(t_p99,  color=C["resid"], lw=1.2, ls=":",
                label=f"p99 = {t_p99:.3f} ms")
axes[0].set_xlabel("Inference time (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"(a) PINN single-sample latency (n={N_REPEATS})", fontweight="bold")
axes[0].legend(fontsize=9); light_grid(axes[0])

# Batch throughput
bs_arr  = list(BATCH_SIZES)
tput    = [batch_results[b]["throughput"] for b in bs_arr]
axes[1].bar(range(len(bs_arr)), tput, color=C["train"], alpha=0.85,
            edgecolor="white", lw=0.4)
axes[1].set_xticks(range(len(bs_arr)))
axes[1].set_xticklabels([str(b) for b in bs_arr])
axes[1].set_xlabel("Batch size")
axes[1].set_ylabel("Throughput (samples/s)")
axes[1].set_title("(b) Batch throughput vs batch size", fontweight="bold")
for i, v in enumerate(tput):
    axes[1].text(i, v + max(tput)*0.01, f"{v:.0f}", ha="center",
                 va="bottom", fontsize=9)
light_grid(axes[1])

fig.suptitle("PINN Inference Latency & Throughput — Device 108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "figCE1_inference_latency.png")


# ── FIG CE-2 — Speedup bar chart vs HYDRUS + EWS ──────────────
fig, ax = plt.subplots(figsize=(10, 5))

categories = ["HYDRUS-1D\n(literature ~60 s)",
              "PIML full pipeline",
              "ML baseline\n(same arch, no physics)",
              "EWS limit\n(real-time threshold)"]

times_sec = [
    HYDRUS_TIME_SEC,
    pipe_sec,
    (t_mean_ml / 1000),   # single sample only (ML has no VG step here)
    EWS_LIMIT_SEC,
]
colors_bar = [C["coral"], C["pred"], C["warn"], "darkorange"]
bars = ax.barh(categories, times_sec, color=colors_bar, alpha=0.85,
               edgecolor="white", lw=0.4, height=0.5)

ax.set_xlabel("Time (seconds)")
ax.set_title("Computational Efficiency: PINN vs HYDRUS-1D vs EWS Requirement\nDevice 108",
             fontweight="bold")
ax.set_xscale("log")

# Annotate
for bar, val in zip(bars, times_sec):
    xpos = bar.get_width() * 1.05
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f"{val:.4f} s" if val < 1 else f"{val:.1f} s",
            va="center", fontsize=10)

ax.axvline(EWS_LIMIT_SEC, color="darkorange", lw=1.5, ls=":",
           label=f"EWS limit = {EWS_LIMIT_SEC} s", alpha=0.7)
ax.legend(fontsize=9); light_grid(ax, axis="x")

# Speedup annotation
ax.text(0.98, 0.12,
        f"PIML pipeline: {speedup_pipe:.0f}× faster than HYDRUS\n"
        f"({pct_faster:.1f}% reduction in compute time)\n"
        f"EWS feasible: {'✅ YES' if pipe_sec < EWS_LIMIT_SEC else '❌ NO'}",
        transform=ax.transAxes, fontsize=10, ha="right", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", lw=0.8))

plt.tight_layout()
savefig(fig, "figCE2_speedup_bar.png")


# ── FIG CE-3 — Pipeline breakdown (stacked bar) ───────────────
fig, ax = plt.subplots(figsize=(9, 4))

# Time breakdown for PIML full pipeline
t_inference_ms = t_mean                      # PINN forward pass
t_vg_ms        = pipe_mean_ms - t_mean       # VG + FoS computation (remainder)
t_vg_ms        = max(t_vg_ms, 0.0)

stages  = ["PINN forward\n(neural net)", "VG + FoS\n(numpy)"]
vals_ms = [t_inference_ms, t_vg_ms]
cols_p  = [C["pred"], C["fos"]]

bars2 = ax.barh(["PIML pipeline"], [t_inference_ms],
                color=C["pred"], alpha=0.85, label=f"PINN forward ({t_inference_ms:.3f} ms)", height=0.4)
ax.barh(["PIML pipeline"], [t_vg_ms], left=[t_inference_ms],
        color=C["fos"], alpha=0.85,
        label=f"VG + FoS ({t_vg_ms:.3f} ms)", height=0.4)
ax.barh(["HYDRUS-1D"], [HYDRUS_TIME_SEC * 1000],
        color=C["coral"], alpha=0.85,
        label=f"HYDRUS-1D ({HYDRUS_TIME_SEC:.0f} s = {HYDRUS_TIME_SEC*1000:.0f} ms)", height=0.4)

ax.set_xlabel("Time (ms, log scale)")
ax.set_xscale("log")
ax.set_title("Pipeline Time Breakdown: PIML vs HYDRUS-1D — Device 108", fontweight="bold")
ax.legend(fontsize=9, loc="lower right"); light_grid(ax, axis="x")
ax.text(0.99, 0.55,
        f"Total PIML pipeline: {pipe_mean_ms:.2f} ms\n"
        f"Speedup vs HYDRUS: {speedup_pipe:.0f}×",
        transform=ax.transAxes, fontsize=10, ha="right",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", lw=0.8))

plt.tight_layout()
savefig(fig, "figCE3_pipeline_breakdown.png")


# ── FIG CE-4 — Per-sample latency vs batch size ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

per_sample = [batch_results[b]["per_sample_ms"] for b in BATCH_SIZES]
axes[0].plot(BATCH_SIZES, per_sample, "o-", color=C["phys"], lw=2, ms=7)
axes[0].set_xlabel("Batch size"); axes[0].set_ylabel("Per-sample latency (ms)")
axes[0].set_title("(a) Per-sample latency vs batch size", fontweight="bold")
axes[0].set_xscale("log", base=2)
for bs, v in zip(BATCH_SIZES, per_sample):
    axes[0].annotate(f"{v:.4f}", (bs, v), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=8)
light_grid(axes[0])

# Latency distribution PIML vs ML
axes[1].hist(times_single, bins=35, alpha=0.7, color=C["pred"],
             label=f"PIML  μ={t_mean:.3f} ms", edgecolor="white", lw=0.3)
axes[1].hist(times_ml,     bins=35, alpha=0.7, color=C["warn"],
             label=f"ML    μ={t_mean_ml:.3f} ms", edgecolor="white", lw=0.3)
axes[1].axvline(t_mean,    color=C["pred"],  lw=2, ls="--")
axes[1].axvline(t_mean_ml, color=C["warn"],  lw=2, ls="--")
axes[1].set_xlabel("Inference time (ms)")
axes[1].set_ylabel("Count")
axes[1].set_title("(b) PIML vs ML latency distribution\n(same architecture, both CPU)", fontweight="bold")
axes[1].legend(fontsize=9); light_grid(axes[1])

fig.suptitle("Inference Latency Analysis — PIML vs ML Baseline — Device 108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "figCE4_latency_analysis.png")


# ── FIG CE-5 — Summary table ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis("off")

col_labels = ["Metric", "PIML (Richards)", "ML Baseline", "HYDRUS-1D", "EWS Limit"]
row_data = [
    ["Single inference (ms)",
     f"{t_mean:.3f} ± {t_std:.3f}",
     f"{t_mean_ml:.3f} ± {times_ml.std():.3f}",
     "~60,000 ms",
     "—"],
    ["Full pipeline (ms)",
     f"{pipe_mean_ms:.2f} ± {pipe_std_ms:.2f}",
     "N/A (no physics step)",
     "~60,000 ms",
     "60,000 ms"],
    ["Throughput (samples/s)",
     f"{1000/t_mean:.0f}",
     f"{1000/t_mean_ml:.0f}",
     "~0.017",
     "—"],
    ["Speedup vs HYDRUS",
     f"{speedup_pipe:.0f}×  ({pct_faster:.1f}% faster)",
     "—",
     "1× (baseline)",
     "—"],
    ["EWS real-time feasible",
     "✅ YES" if pipe_sec < EWS_LIMIT_SEC else "❌ NO",
     "✅ YES (data only)",
     "❌ NO",
     f"< {EWS_LIMIT_SEC:.0f} s required"],
    ["Physics PDE enforced",
     "✅ Richards equation",
     "❌ None",
     "✅ Richards equation",
     "—"],
    ["Device",
     str(DEVICE).upper(),
     str(DEVICE).upper(),
     "CPU (desktop)",
     "—"],
]

tbl = ax.table(cellText=row_data, colLabels=col_labels,
               cellLoc="center", loc="center",
               colWidths=[0.28, 0.20, 0.18, 0.18, 0.16])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 2.0)

for j in range(5):
    tbl[0, j].set_facecolor("#023E8A")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

highlight = {2, 4}
for i in range(1, len(row_data)+1):
    for j in range(5):
        if i in highlight:
            tbl[i, j].set_facecolor("#E3F2FD")
        elif i % 2 == 0:
            tbl[i, j].set_facecolor("#F8F9FA")
        else:
            tbl[i, j].set_facecolor("white")

ax.set_title(
    f"Table CE — Computational Efficiency: PIML vs ML vs HYDRUS-1D vs EWS Requirement\n"
    f"Device 108  ·  CPU: {platform.machine()}  ·  N={N_REPEATS} timed runs",
    fontsize=11, fontweight="bold", pad=18)
plt.tight_layout()
savefig(fig, "figCE5_efficiency_table.png")


# ── Print final summary ────────────────────────────────────────
print("\n" + "="*65)
print("  COMPUTATIONAL EFFICIENCY SUMMARY — Device 108")
print("="*65)
print(f"  Single inference   : {t_mean:.3f} ± {t_std:.3f} ms")
print(f"  Full pipeline      : {pipe_mean_ms:.2f} ± {pipe_std_ms:.2f} ms  ({pipe_sec:.4f} s)")
print(f"  Throughput         : {1000/t_mean:.0f} samples/s")
print(f"  Speedup vs HYDRUS  : {speedup_pipe:.0f}×  ({pct_faster:.1f}% faster)")
print(f"  EWS feasible (<60s): {'✅ YES' if pipe_sec < EWS_LIMIT_SEC else '❌ NO'}")
print(f"\n✅ Figures → {OUT}/")
print("   figCE1 — Latency distribution + batch throughput")
print("   figCE2 — Speedup bar chart vs HYDRUS + EWS")
print("   figCE3 — Pipeline breakdown (stacked)")
print("   figCE4 — Per-sample latency + PIML vs ML comparison")
print("   figCE5 — Summary table (publication-ready)")


Device: cpu

[Step 1-2] Loading data for Device 108 ...
  Rows after subsample: 30,977

[Step 3] Building PINN (true autograd Richards) ...
  Parameters: 43,777

[Step 4-6] Training PINN with true Richards autograd loss ...
  ∂θ/∂t scale for normalization: 8.7802e-08 m³/m³/s
  Epoch  100 | Data 0.00283 | Phys(norm) 0.00002 | λ·Phys/Data ratio: 0.039 | Val 0.00052 | LR 5.00e-04
  Epoch  200 | Data 0.00284 | Phys(norm) 0.00001 | λ·Phys/Data ratio: 0.015 | Val 0.00103 | LR 4.52e-04
  Epoch  300 | Data 0.00026 | Phys(norm) 0.00004 | λ·Phys/Data ratio: 0.786 | Val 0.00018 | LR 3.27e-04
  Epoch  400 | Data 0.00008 | Phys(norm) 0.00001 | λ·Phys/Data ratio: 0.504 | Val 0.00003 | LR 1.73e-04
  Epoch  500 | Data 0.00006 | Phys(norm) 0.00000 | λ·Phys/Data ratio: 0.094 | Val 0.00002 | LR 4.77e-05
  Epoch  600 | Data 0.00005 | Phys(norm) 0.00000 | λ·Phys/Data ratio: 0.034 | Val 0.00002 | LR 0.00e+00

  Best val: 0.00002  @ epoch 382

[Model Save] Saving model ...
  → Model saved: /content/piml_figu

In [2]:
# ═══════════════════════════════════════════════════════════════
# ADD-ON CELL — 4-Model Benchmark  [SCIENTIFICALLY CORRECTED]
#
# FIX 1 → LSTM: unsqueeze(1) সরানো, proper (B, LAG, 2) sequence
# FIX 2 → HYDRUS: steady-state VG inversion → transient Euler stepping
# FIX 3 → λ=5.0: hardcoded নয়, grid search দিয়ে select করা হবে
#
# Paste this in a NEW cell AFTER your existing script has run.
# Already uses: model (PINN), model_ml (MLP), all data splits,
#               scalers, VG params, FoS params from existing scope.
# ═══════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════
# PRE-STEP: λ GRID SEARCH  [FIX 3]
# LAM=5.0 ছিল hardcoded — এখন validation MSE দিয়ে select হবে
# ~10-15 min on CPU (200 epoch × 5 candidates)
# ══════════════════════════════════════════════════════════════
print("█"*60)
print("  PRE-STEP: λ grid search (200 epoch × 5 candidates)")
print("  This justifies LAM selection scientifically.")
print("█"*60)

LAM_CANDIDATES = [0.5, 1.0, 5.0, 10.0, 50.0]
lam_val_results = {}

for lam_try in LAM_CANDIDATES:
    torch.manual_seed(42); np.random.seed(42)
    m_try  = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
    opt_try = optim.AdamW(m_try.parameters(), lr=LR, weight_decay=1e-4)
    sch_try = optim.lr_scheduler.LambdaLR(opt_try, lr_lambda)
    best_v  = np.inf

    for ep in range(1, 201):
        m_try.train()
        for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
            opt_try.zero_grad()
            B   = Xb.shape[0]
            z_d = z_sens_norm.expand(B, -1).requires_grad_(True)
            t_d = tb_norm.clone().requires_grad_(True)
            th  = m_try(z_d, t_d, Xb)
            pred_norm = (th - y_min_t) * y_scl_t
            loss_d    = mse_loss(pred_norm, yb)

            tb_sec_np = tb_sec.detach().cpu().numpy().flatten()
            z_c, t_c, feat_c = make_colloc_batch(tb_sec_np, None,
                                                  Xb.detach().cpu().numpy())
            resid, _, _ = richards_residual(m_try, z_c, t_c, feat_c)
            resid_norm  = resid / (DTHDT_SCALE + 1e-12)
            loss_p      = (resid_norm ** 2).mean()

            (loss_d + lam_try * loss_p).backward()
            torch.nn.utils.clip_grad_norm_(m_try.parameters(), 1.0)
            opt_try.step()
        sch_try.step()

        m_try.eval()
        with torch.no_grad():
            vl = mse_loss(
                (m_try(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t, yv
            ).item()
        if vl < best_v:
            best_v = vl

    lam_val_results[lam_try] = best_v
    print(f"  λ = {lam_try:5.1f}  →  best val MSE = {best_v:.6f}")

# Best λ select করো
LAM_SELECTED = min(lam_val_results, key=lam_val_results.get)
print(f"\n  ★ Selected λ = {LAM_SELECTED}  "
      f"(val MSE = {lam_val_results[LAM_SELECTED]:.6f})")
print(f"  (Previously hardcoded LAM = {LAM}; "
      f"{'same' if LAM_SELECTED == LAM else 'DIFFERENT — update LAM!'} )")

# Paper-এ এই table দাও:
print("\n  λ sensitivity table (for paper Methods section):")
print(f"  {'λ':<8} {'Best Val MSE':>14}")
for k, v in sorted(lam_val_results.items()):
    marker = " ← selected" if k == LAM_SELECTED else ""
    print(f"  {k:<8.1f} {v:>14.6f}{marker}")


# ══════════════════════════════════════════════════════════════
# STEP A: LSTM Definition & Training  [FIX 1]
#
# পুরানো সমস্যা:
#   self.lstm = nn.LSTM(input_size=feat_dim, ...)  # feat_dim=16
#   x_seq = feat_flat.unsqueeze(1)                 # (B,1,16): SINGLE timestep
#   → LSTM recurrent memory কাজ করছিল না
#
# সমাধান:
#   input_size=2 (rain + theta per step)
#   sequence shape: (B, LAG, 2)  ← LAG=15 timesteps
# ══════════════════════════════════════════════════════════════

# ── LSTM-এর জন্য proper 3D sequence builder ──────────────────
def build_lstm_sequences(X_raw_norm, lag=LAG):
    """
    X_raw_norm : (N, 1+lag) — col0=rain_norm, col1..lag=theta_lags_norm
    Returns    : (N, lag, 2) — proper temporal sequence
                 dim0=timestep, dim1=[rain, theta_at_lag_k]
    """
    N   = X_raw_norm.shape[0]
    seq = np.zeros((N, lag, 2), dtype=np.float32)
    for k in range(lag):
        seq[:, k, 0] = X_raw_norm[:, 0]      # rain (same context at each step)
        seq[:, k, 1] = X_raw_norm[:, k + 1]  # lagged θ, newest first
    return seq

X_tr_lstm  = build_lstm_sequences(X_tr)    # (N_tr,  15, 2)
X_val_lstm = build_lstm_sequences(X_val)   # (N_val, 15, 2)
X_te_lstm  = build_lstm_sequences(X_te)    # (N_te,  15, 2)
X_all_lstm = build_lstm_sequences(X_norm)  # (N_all, 15, 2)

# Shape assertion — fail করলে dataloader ঠিক নেই
assert X_tr_lstm.shape == (len(X_tr), LAG, 2), \
    f"LSTM input shape wrong: got {X_tr_lstm.shape}, expected {(len(X_tr), LAG, 2)}"
print(f"\n  [FIX 1] LSTM input shape verified: {X_tr_lstm.shape}  ✓")
print(f"          (was: (N, 1, {1+LAG}) — single timestep, recurrent memory unused)")
print(f"          (now: (N, {LAG}, 2)  — {LAG} timesteps, each with [rain, θ])")


class SoilLSTM(nn.Module):
    """
    1-layer LSTM on proper (B, LAG, 2) sequence input.
    [FIX 1]: input_size=2, NOT feat_dim=16.
    Hidden=64, 3-layer head — matches original parameter budget.
    """
    def __init__(self, seq_features=2, h=64, num_layers=1):
        super().__init__()
        # [FIX 1] input_size=2, NOT the flat feat_dim
        self.lstm = nn.LSTM(input_size=seq_features, hidden_size=h,
                            num_layers=num_layers, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(h, h),       nn.Tanh(),
            nn.Linear(h, h // 2),  nn.Tanh(),
            nn.Linear(h // 2, 1),  nn.Sigmoid(),
        )
        self.theta_lo = VG["theta_r"] + 0.01
        self.theta_hi = VG["theta_s"] - 0.01

    def forward(self, x_seq):
        # [FIX 1] x_seq: (B, LAG, 2) — proper temporal sequence
        # NOT: feat_flat.unsqueeze(1) which gave (B, 1, 16)
        out, _  = self.lstm(x_seq)   # (B, LAG, H)
        h_last  = out[:, -1, :]      # last timestep hidden state: (B, H)
        raw     = self.head(h_last)
        return self.theta_lo + (self.theta_hi - self.theta_lo) * raw


print("\n" + "█"*60)
print("  Training LSTM (same: epochs, LR, schedule, batch, seed)")
print("  [FIX 1] proper sequence input: (B, 15, 2)")
print("█"*60)

torch.manual_seed(42); np.random.seed(42)
model_lstm = SoilLSTM(seq_features=2).to(DEVICE)
print(f"  LSTM params : {sum(p.numel() for p in model_lstm.parameters()):,}")
print(f"  PINN params : {sum(p.numel() for p in model.parameters()):,}")
print(f"  MLP  params : {sum(p.numel() for p in model_ml.parameters()):,}")

# [FIX 1] X_tr_lstm (3D sequence) দিয়ে DataLoader
lstm_dataset = TensorDataset(to_t(X_tr_lstm), to_t(y_tr))
lstm_loader  = DataLoader(lstm_dataset, batch_size=BATCH, shuffle=False)

opt_lstm   = optim.AdamW(model_lstm.parameters(), lr=LR, weight_decay=1e-4)
sched_lstm = optim.lr_scheduler.LambdaLR(opt_lstm, lr_lambda)

lstm_train_losses, lstm_val_losses = [], []
best_val_lstm, best_state_lstm = np.inf, None

Xv_lstm = to_t(X_val_lstm)  # validation도 3D

for epoch in range(1, EPOCHS + 1):
    model_lstm.train()
    ep = 0.0
    for (Xb_seq, yb) in lstm_loader:
        opt_lstm.zero_grad()
        pred = model_lstm(Xb_seq)          # [FIX 1] 3D sequence input
        loss = mse_loss((pred - y_min_t) * y_scl_t, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)
        opt_lstm.step()
        ep += loss.item()

    sched_lstm.step()
    model_lstm.eval()
    with torch.no_grad():
        vl = mse_loss(
            (model_lstm(Xv_lstm) - y_min_t) * y_scl_t, yv
        ).item()

    lstm_train_losses.append(ep / len(lstm_loader))
    lstm_val_losses.append(vl)
    if vl < best_val_lstm:
        best_val_lstm   = vl
        best_state_lstm = {k: v.clone() for k, v in model_lstm.state_dict().items()}
    if epoch % 100 == 0:
        print(f"  [LSTM] Epoch {epoch:4d} | train={ep/len(lstm_loader):.5f} | val={vl:.5f}")

model_lstm.load_state_dict(best_state_lstm)
best_ep_lstm = int(np.argmin(lstm_val_losses)) + 1
print(f"\n  [LSTM] Best val={best_val_lstm:.5f}  @ epoch {best_ep_lstm}")


# ══════════════════════════════════════════════════════════════
# STEP B: HYDRUS-1D Proxy  [FIX 2]
#
# পুরানো সমস্যা:
#   K(θ_eq) = rain_flux → steady-state VG inversion
#   প্রতিটা timestep independently compute হচ্ছিল
#   τ=12 hardcoded, কোনো justification নেই
#   wetting front, transient infiltration model করতে পারে না
#
# সমাধান:
#   Explicit Euler time-stepping দিয়ে dθ/dt integrate করা
#   Physical water balance: q_in (rainfall) - q_drain (gravity) per layer
#   Label পরিবর্তন: "simplified transient proxy" (honest)
#   τ calibration loop: observed θ-এর বিপরীতে RMSE minimize করে
# ══════════════════════════════════════════════════════════════

def hydrus_proxy_transient(rain_arr, theta_init_arr, t_sec_arr, vg=VG, z=Z_SENSOR):
    """
    [FIX 2] Simplified 1-layer transient Richards proxy via explicit Euler.

    Replaces steady-state VG inversion with proper time-stepping:
      dθ/dt = (q_in - q_drain) / dz
    where:
      q_in    = min(rainfall_flux, Ks)   [infiltration capacity limit]
      q_drain = K(θ)                     [gravity drainage, unit gradient]
      dz      = sensor depth (Z_SENSOR)  [single-layer thickness]

    Initial condition: theta_init_arr[0] only (not used at later steps).
    Label in paper: "simplified single-layer transient proxy"
    """
    N     = len(rain_arr)
    theta = np.zeros(N, dtype=np.float64)
    theta[0] = float(theta_init_arr[0])   # IC: only first timestep

    for i in range(1, N):
        dt = float(t_sec_arr[i] - t_sec_arr[i - 1])
        if dt <= 0:
            dt = 1.0

        th_prev = np.clip(theta[i - 1],
                          vg["theta_r"] + 1e-6,
                          vg["theta_s"] - 1e-6)

        # K(θ) at previous state
        K = vg_K_np(np.array([th_prev]), vg)[0]   # m/s

        # Rainfall flux: mm/min → m/s
        q_rain = (float(rain_arr[i]) / 1000.0) / 60.0

        # Infiltration limited by Ks (Horton-type cap)
        q_in = min(q_rain, vg["Ks_ms"])

        # Gravity drainage (unit hydraulic gradient → q = K)
        q_drain = K

        # Single-layer water balance
        dz     = z                                   # m
        dtheta = (q_in - q_drain) * dt / dz

        theta[i] = np.clip(th_prev + dtheta,
                            vg["theta_r"] + 1e-6,
                            vg["theta_s"] - 1e-6)

    return theta.astype(np.float32)


# ── τ calibration (replaces hardcoded τ=12) ───────────────────
# Training split-এ RMSE minimize করে τ select করা হচ্ছে
print("\n  [FIX 2] Calibrating τ (exponential smoothing) against training θ ...")
print("  (Previously τ=12 was hardcoded with no justification)")

TAU_CANDIDATES = [1, 3, 6, 12, 24, 48]
tau_rmse = {}

def hydrus_with_tau(rain_arr, theta_init_arr, t_sec_arr, tau, vg=VG, z=Z_SENSOR):
    """Transient proxy + optional exponential smoothing with given τ."""
    raw = hydrus_proxy_transient(rain_arr, theta_init_arr, t_sec_arr, vg, z)
    if tau == 0:
        return raw
    smoothed = np.zeros_like(raw)
    smoothed[0] = raw[0]
    alpha = 1.0 - np.exp(-1.0 / tau)
    for i in range(1, len(raw)):
        smoothed[i] = alpha * raw[i] + (1.0 - alpha) * smoothed[i - 1]
    return smoothed

for tau_try in TAU_CANDIDATES:
    pred_tau = hydrus_with_tau(rain_tr, theta_tr, t_sec_tr, tau=tau_try)
    rmse_tau = np.sqrt(np.mean((theta_tr - pred_tau) ** 2))
    tau_rmse[tau_try] = rmse_tau
    print(f"  τ = {tau_try:3d}  →  train RMSE = {rmse_tau:.5f} m³/m³")

TAU_SELECTED = min(tau_rmse, key=tau_rmse.get)
print(f"\n  ★ Selected τ = {TAU_SELECTED}  "
      f"(train RMSE = {tau_rmse[TAU_SELECTED]:.5f})")
print(f"  (Previously hardcoded τ=12; "
      f"{'same' if TAU_SELECTED == 12 else 'DIFFERENT — update τ!'} )")

# Final proxy with calibrated τ
def hydrus_proxy(rain_arr, theta_init_arr, t_sec_arr,
                 tau=TAU_SELECTED, vg=VG, z=Z_SENSOR):
    """
    [FIX 2] Production HYDRUS proxy: transient Euler + calibrated smoothing.
    Call signature updated: now requires t_sec_arr.
    """
    return hydrus_with_tau(rain_arr, theta_init_arr, t_sec_arr, tau=tau, vg=vg, z=z)

print("\n  [HYDRUS] Running transient proxy on all splits ...")


# ══════════════════════════════════════════════════════════════
# STEP C: Predict all models on all splits
# [FIX 1] pred_lstm এখন 3D sequence নেয়
# [FIX 2] hydrus_proxy এখন t_sec_arr নেয়
# ══════════════════════════════════════════════════════════════
def pred_lstm(X_raw_norm_2d):
    """[FIX 1] Converts flat (N,16) → sequence (N,15,2) before inference."""
    model_lstm.eval()
    seq = build_lstm_sequences(X_raw_norm_2d)   # (N, LAG, 2)
    with torch.no_grad():
        return model_lstm(to_t(seq)).cpu().numpy().flatten()

# LSTM predictions
lstm_tr  = pred_lstm(X_tr)
lstm_val = pred_lstm(X_val)
lstm_te  = pred_lstm(X_te)
lstm_all = pred_lstm(X_norm)

# HYDRUS predictions  [FIX 2: t_sec_arr argument added]
hyd_tr   = hydrus_proxy(rain_tr,  theta_tr,  t_sec_tr)
hyd_val  = hydrus_proxy(rain_val, theta_val, t_sec_val)
hyd_te   = hydrus_proxy(rain_te,  theta_te,  t_sec_te)
hyd_all  = hydrus_proxy(rain_np,  theta_np,  t_sec_np)


# ══════════════════════════════════════════════════════════════
# STEP D: Metrics for all 4 models
# ══════════════════════════════════════════════════════════════
from sklearn.metrics import r2_score, mean_squared_error

def get_metrics(obs_tr, obs_val, obs_te, pr_tr, pr_val, pr_te):
    return {
        "r2_tr"   : r2_score(obs_tr,  pr_tr),
        "r2_val"  : r2_score(obs_val, pr_val),
        "r2_te"   : r2_score(obs_te,  pr_te),
        "rmse_te" : np.sqrt(mean_squared_error(obs_te, pr_te)),
        "mae_te"  : np.mean(np.abs(obs_te - pr_te)),
        "resid_te": obs_te - pr_te,
    }

def get_fos(theta_arr):
    psi_raw = vg_psi_np(theta_arr, VG)
    psi     = np.clip(psi_raw, np.percentile(psi_raw, 2), 0.0)
    u       = gamma_w * psi
    FoS     = (c_ + np.maximum(sigma_n - u, 0) * np.tan(phi_)) / (tau_d + 1e-8)
    return FoS, u

bench = {
    "PINN"  : get_metrics(theta_tr, theta_val, theta_te,
                          theta_pred_tr, theta_pred_val, theta_pred_te),
    "LSTM"  : get_metrics(theta_tr, theta_val, theta_te,
                          lstm_tr, lstm_val, lstm_te),
    "MLP"   : get_metrics(theta_tr, theta_val, theta_te,
                          theta_ml_tr, theta_ml_val, theta_ml_te),
    "HYDRUS": get_metrics(theta_tr, theta_val, theta_te,
                          hyd_tr, hyd_val, hyd_te),
}

for mname in bench:
    FoS, u = get_fos({"PINN"  : theta_pred_te,
                       "LSTM"  : lstm_te,
                       "MLP"   : theta_ml_te,
                       "HYDRUS": hyd_te}[mname])
    bench[mname].update({
        "fos_mean": FoS.mean(), "fos_min": FoS.min(), "fos_std": FoS.std(),
        "n_fail"  : int((FoS < 1.0).sum()),
        "n_warn"  : int((FoS < 1.3).sum()),
        "fos_all" : FoS, "u_all": u,
    })

def approx_pde_resid(theta_arr, t_sec_arr):
    dth_dt = np.gradient(theta_arr, t_sec_arr - t_sec_arr[0] + 1e-8) / (T_MAX - T_MIN)
    K_arr  = vg_K_np(theta_arr, VG)
    dflux  = np.gradient(K_arr) / Z_MAX
    return dth_dt - dflux

pde_all = {
    "PINN"  : phys_residual,
    "LSTM"  : approx_pde_resid(lstm_all,  t_sec_np),
    "MLP"   : phys_residual_ml,
    "HYDRUS": approx_pde_resid(hyd_all,   t_sec_np),
}
for m in bench:
    bench[m]["pde_mu"] = pde_all[m].mean()
    bench[m]["pde_sd"] = pde_all[m].std()


# ══════════════════════════════════════════════════════════════
# STEP E: Inference Speed
# [FIX 2] hydrus speed benchmark: t_sec_arr argument যোগ
# ══════════════════════════════════════════════════════════════
import timeit
N_W, N_R       = 20, 200
HYDRUS_LIT_SEC = 60.0

def bench_speed(fn):
    for _ in range(N_W): fn()
    times = []
    for _ in range(N_R):
        t0 = timeit.default_timer()
        fn()
        times.append(timeit.default_timer() - t0)
    return np.array(times) * 1000

speed = {}
speed["PINN"]   = bench_speed(
    lambda: predict_sensor(X_norm[-1:], t_sec_np[-1:]))
speed["LSTM"]   = bench_speed(
    lambda: pred_lstm(X_norm[-1:]))
speed["MLP"]    = bench_speed(
    lambda: predict_sensor_ml(X_norm[-1:], t_sec_np[-1:]))
speed["HYDRUS"] = bench_speed(
    # [FIX 2] t_sec_np argument added
    lambda: hydrus_proxy(rain_np[-1:], theta_np[-1:], t_sec_np[-1:]))

for m in bench:
    bench[m]["lat_mean_ms"] = speed[m].mean()
    bench[m]["lat_std_ms"]  = speed[m].std()

print("\n  Inference latency (single sample):")
for m in bench:
    mu = speed[m].mean()
    print(f"  [{m:6s}] {mu:.3f} ± {speed[m].std():.3f} ms  "
          f"| speedup vs HYDRUS: {HYDRUS_LIT_SEC/(mu/1000):.0f}×")


# ══════════════════════════════════════════════════════════════
# FIGURES — unchanged from original (C1–C9)
# Only model descriptions updated to reflect fixes
# ══════════════════════════════════════════════════════════════
MODEL_NAMES = ["PINN", "LSTM", "MLP", "HYDRUS"]
COLORS = {"PINN": "#2EC4B6", "LSTM": "#F77F00", "MLP": "#EF233C", "HYDRUS": "#06D6A0"}
LS     = {"PINN": "-",       "LSTM": "--",       "MLP": ":",       "HYDRUS": "-."}
LABELS = {
    "PINN"  : "PINN (Richards)",
    "LSTM"  : "LSTM (seq input)",          # updated: proper sequence
    "MLP"   : "Vanilla MLP",
    "HYDRUS": "Transient Proxy",           # updated: no longer "steady-state"
}

preds_all = {"PINN": theta_pred_all, "LSTM": lstm_all,
             "MLP": theta_ml_all,    "HYDRUS": hyd_all}
preds_te  = {"PINN": theta_pred_te,  "LSTM": lstm_te,
             "MLP": theta_ml_te,     "HYDRUS": hyd_te}

n_tr_p     = len(theta_tr)
n_va_p     = n_tr_p + len(theta_val)
ts_te_plot = ts[n_va_p:]

epochs_arr = np.arange(1, EPOCHS + 1)

# ── FIG C1 — Loss Convergence ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for mname, lhist, col in [
    ("PINN", train_losses,      COLORS["PINN"]),
    ("LSTM", lstm_train_losses, COLORS["LSTM"]),
    ("MLP",  train_losses_ml,   COLORS["MLP"]),
]:
    axes[0].semilogy(epochs_arr, lhist, color=col, lw=1.8, label=f"{mname} train")
axes[0].set_title("(A) Training Loss"); axes[0].set_ylabel("MSE Loss")
axes[0].legend(fontsize=8); light_grid(axes[0])

for mname, lhist, col in [
    ("PINN", val_losses,      COLORS["PINN"]),
    ("LSTM", lstm_val_losses, COLORS["LSTM"]),
    ("MLP",  val_losses_ml,   COLORS["MLP"]),
]:
    axes[1].semilogy(epochs_arr, lhist, color=col, lw=1.8, ls="--", label=f"{mname} val")
    best_e = int(np.argmin(lhist)) + 1
    axes[1].axvline(best_e, color=col, lw=0.7, ls=":", alpha=0.5)
axes[1].set_title("(B) Validation Loss"); axes[1].set_ylabel("MSE Loss")
axes[1].legend(fontsize=8); light_grid(axes[1])

best_vals = [min(val_losses), min(lstm_val_losses), min(val_losses_ml)]
colors_b  = [COLORS["PINN"], COLORS["LSTM"], COLORS["MLP"]]
bars = axes[2].bar(["PINN", "LSTM", "MLP"], best_vals, color=colors_b,
                   alpha=0.85, edgecolor="white", lw=0.4)
for bar, v in zip(bars, best_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, v*1.05, f"{v:.5f}",
                 ha="center", va="bottom", fontsize=9)
axes[2].set_yscale("log"); axes[2].set_ylabel("Best Val MSE Loss")
axes[2].set_title("(C) Best Validation Loss"); light_grid(axes[2])

fig.suptitle(
    f"Training Convergence — 4-Model Benchmark  Device 108\n"
    f"Same: optimizer(AdamW) · LR-schedule(cosine) · epochs({EPOCHS}) · "
    f"batch({BATCH}) · seed(42)  |  λ={LAM_SELECTED} (grid-searched)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
savefig(fig, "figC1_loss_convergence_4model.png")


# ── FIG C2 — Scatter R² ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for idx, mname in enumerate(MODEL_NAMES):
    ax   = axes[idx//2][idx%2]
    obs  = theta_te
    pred = preds_te[mname]
    col  = COLORS[mname]
    r2   = bench[mname]["r2_te"]
    rmse = bench[mname]["rmse_te"]
    ax.scatter(obs, pred, s=6, alpha=0.3, color=col, rasterized=True)
    lim = [min(obs.min(), pred.min())-0.003, max(obs.max(), pred.max())+0.003]
    ax.plot(lim, lim, "k--", lw=1.2, label="1:1")
    m_ols, b_ols = np.polyfit(obs, pred, 1)
    ax.plot(np.linspace(*lim, 100), m_ols*np.linspace(*lim, 100)+b_ols,
            color=col, lw=1.5, label="OLS")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("θ Observed (m³/m³)"); ax.set_ylabel("θ Predicted (m³/m³)")
    ax.set_title(f"{LABELS[mname]}\nTest R²={r2:.4f}  RMSE={rmse:.4f}")
    ax.legend(fontsize=8); light_grid(ax)
fig.suptitle("Scatter R² — 4-Model Benchmark (Test Set)  Device 108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "figC2_scatter_r2_4model.png")


# ── FIG C3 — θ Time-series ───────────────────────────────────
fig, axes = plt.subplots(6, 1, figsize=(15, 17), sharex=True,
                         gridspec_kw={"height_ratios":[0.8,2,2,2,2,2], "hspace":0.10})
axes[0].fill_between(ts, rain_np, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rain\n(mm/min)"); light_grid(axes[0])
axes[0].set_title("A — Rainfall", fontweight="bold", loc="left")
axes[1].plot(ts, theta_obs_all, color="#333333", lw=1.1, label="Observed θ")
axes[1].set_ylabel("θ Obs"); axes[1].set_title("B — Observed", fontweight="bold", loc="left")
axes[1].legend(fontsize=8); light_grid(axes[1])
for i, mname in enumerate(MODEL_NAMES):
    ax   = axes[i + 2]
    col  = COLORS[mname]
    pred = preds_all[mname]
    r2   = bench[mname]["r2_te"]
    rmse = bench[mname]["rmse_te"]
    ax.plot(ts, theta_obs_all, color="#333333", lw=0.7, alpha=0.45, label="Observed")
    ax.plot(ts, pred, color=col, lw=1.2, ls=LS[mname],
            label=f"{LABELS[mname]}  R²={r2:.4f}  RMSE={rmse:.4f}")
    ax.axvspan(ts[0],         ts[n_tr_p-1], alpha=0.04, color=C["train"])
    ax.axvspan(ts[n_tr_p-1],  ts[n_va_p-1], alpha=0.05, color=C["val"])
    ax.axvspan(ts[n_va_p-1],  ts[-1],        alpha=0.05, color=C["test"])
    ax.set_ylabel(f"θ — {mname}")
    ax.set_title(f"{chr(67+i)} — {LABELS[mname]}", fontweight="bold", loc="left")
    ax.legend(fontsize=8, ncol=2); light_grid(ax)
for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for tick in ax.get_xticklabels(): tick.set_rotation(15)
axes[-1].set_xlabel("Timestamp")
fig.suptitle("θ Time-Series — 4-Model Benchmark  Device 108",
             fontsize=12, fontweight="bold")
savefig(fig, "figC3_theta_timeseries_4model.png")


# ── FIG C4 — FoS Time-series ─────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(15, 14), sharex=True,
                         gridspec_kw={"height_ratios":[0.8,2,2,2,2], "hspace":0.10})
axes[0].fill_between(ts_te_plot, rain_te, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rain\n(mm/min)"); light_grid(axes[0])
axes[0].set_title("A — Rainfall (Test)", fontweight="bold", loc="left")
fos_min_all = min(bench[m]["fos_min"] for m in MODEL_NAMES)
for i, mname in enumerate(MODEL_NAMES):
    ax  = axes[i + 1]
    col = COLORS[mname]
    fos = bench[mname]["fos_all"]
    ax.plot(ts_te_plot, fos, color=col, lw=1.3,
            label=f"{LABELS[mname]}  min={bench[mname]['fos_min']:.3f}  "
                  f"fail={bench[mname]['n_fail']}")
    ax.axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
    ax.axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3")
    ax.fill_between(ts_te_plot, fos, 1.0, where=(fos < 1.0),
                    color=C["warn"], alpha=0.35)
    ax.set_ylabel(f"FoS — {mname}")
    ax.set_title(f"{chr(66+i)} — {LABELS[mname]}", fontweight="bold", loc="left")
    ax.legend(fontsize=8, ncol=3); light_grid(ax)
    ax.set_ylim(bottom=max(0, fos_min_all - 0.1))
for ax in axes:
    ax.set_xlim(ts_te_plot[0], ts_te_plot[-1])
    for tick in ax.get_xticklabels(): tick.set_rotation(15)
axes[-1].set_xlabel("Timestamp (Test Set)")
fig.suptitle("Factor of Safety — 4-Model Benchmark  Device 108",
             fontsize=12, fontweight="bold")
savefig(fig, "figC4_fos_timeseries_4model.png")


# ── FIG C5 — R²/RMSE + FoS bars ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cols_b = [COLORS[m] for m in MODEL_NAMES]
r2s   = [bench[m]["r2_te"]   for m in MODEL_NAMES]
rmses = [bench[m]["rmse_te"] for m in MODEL_NAMES]
fos_mins = [bench[m]["fos_min"] for m in MODEL_NAMES]
for ax, vals, ylabel, title, hb in [
    (axes[0], r2s,    "Test R² (−)",           "(A) Test R²  [higher = better]",  True),
    (axes[1], rmses,  "Test RMSE (m³/m³)",     "(B) Test RMSE  [lower = better]", False),
    (axes[2], fos_mins, "FoS Min (−)",          "(C) FoS Minimum  [higher = better]", True),
]:
    bars = ax.bar(MODEL_NAMES, vals, color=cols_b, alpha=0.85, edgecolor="white", lw=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v*(1.01 if hb else 1.01),
                f"{v:.4f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel(ylabel); ax.set_title(title); light_grid(ax)
axes[0].set_ylim(0, 1.05)
axes[2].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[2].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warn")
axes[2].legend(fontsize=8)
fig.suptitle("Key Metrics Comparison — 4-Model Benchmark  Device 108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "figC5_metric_bars.png")


# ── FIG C6 — Radar Chart ──────────────────────────────────────
def norm01(vals, higher_better=True):
    mn, mx = min(vals), max(vals)
    if mx == mn: return [0.5]*len(vals)
    n = [(v-mn)/(mx-mn) for v in vals]
    return n if higher_better else [1-v for v in n]

cats = ["Test R²", "RMSE↓", "FoS Min", "PDE σ↓", "Speed↑"]
raw  = {
    "Test R²" : ([bench[m]["r2_te"]        for m in MODEL_NAMES], True),
    "RMSE↓"   : ([bench[m]["rmse_te"]      for m in MODEL_NAMES], False),
    "FoS Min" : ([bench[m]["fos_min"]      for m in MODEL_NAMES], True),
    "PDE σ↓"  : ([bench[m]["pde_sd"]       for m in MODEL_NAMES], False),
    "Speed↑"  : ([1/(speed[m].mean()+1e-6) for m in MODEL_NAMES], True),
}
norm_data = {m: [] for m in MODEL_NAMES}
for cat in cats:
    vals, hb = raw[cat]
    normed   = norm01(vals, hb)
    for i, m in enumerate(MODEL_NAMES):
        norm_data[m].append(normed[i])
N_cats = len(cats)
angles = np.linspace(0, 2*np.pi, N_cats, endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for mname in MODEL_NAMES:
    vals = norm_data[mname] + norm_data[mname][:1]
    ax.plot(angles, vals, "o-", color=COLORS[mname], lw=2, ms=7, label=LABELS[mname])
    ax.fill(angles, vals, alpha=0.10, color=COLORS[mname])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=12)
ax.set_ylim(0, 1); ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25","0.5","0.75","1.0"], fontsize=7, alpha=0.5)
ax.set_title("Normalised Performance Radar\n(all metrics 0→1, higher=better)",
             fontsize=12, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.38, 1.18), fontsize=10)
plt.tight_layout()
savefig(fig, "figC6_radar_4model.png")


# ── FIG C7 — Speed boxplot + speedup ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bp = axes[0].boxplot([speed[m] for m in MODEL_NAMES], labels=MODEL_NAMES,
                     patch_artist=True, medianprops=dict(color="black", lw=2))
for patch, col in zip(bp["boxes"], [COLORS[m] for m in MODEL_NAMES]):
    patch.set_facecolor(col); patch.set_alpha(0.7)
axes[0].set_ylabel("Single-sample inference (ms)")
axes[0].set_title("(A) Latency Distribution"); light_grid(axes[0])
for i, m in enumerate(MODEL_NAMES):
    axes[0].text(i+1, speed[m].mean()*1.05, f"{speed[m].mean():.3f}",
                 ha="center", fontsize=8)
speedups = {m: HYDRUS_LIT_SEC / (speed[m].mean() / 1000) for m in MODEL_NAMES}
axes[1].bar(list(speedups.keys()), list(speedups.values()),
            color=[COLORS[m] for m in MODEL_NAMES], alpha=0.85, edgecolor="white", lw=0.4)
axes[1].set_yscale("log"); axes[1].set_ylabel("Speedup vs HYDRUS-1D (×)")
axes[1].set_title(f"(B) Speedup vs HYDRUS-1D literature (~{HYDRUS_LIT_SEC:.0f} s)")
for i, (m, v) in enumerate(speedups.items()):
    axes[1].text(i, v*1.5, f"{v:.0f}×", ha="center", fontsize=11, fontweight="bold")
light_grid(axes[1])
fig.suptitle("Computational Efficiency — 4-Model Benchmark  Device 108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "figC7_speed_4model.png")


# ── FIG C8 — PDE Residual histograms ─────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(17, 5))
for i, mname in enumerate(MODEL_NAMES):
    resid = pde_all[mname]
    col   = COLORS[mname]
    sd    = bench[mname]["pde_sd"]
    mu    = bench[mname]["pde_mu"]
    axes[i].hist(resid, bins=50, color=col, edgecolor="white", lw=0.3, alpha=0.85)
    axes[i].axvline(0,   color="black", lw=1.0, ls="--")
    axes[i].axvline( sd, color="gray",  lw=1.2, ls=":", label=f"σ={sd:.2e}")
    axes[i].axvline(-sd, color="gray",  lw=1.2, ls=":")
    axes[i].set_title(f"{LABELS[mname]}\nμ={mu:.2e}  σ={sd:.2e}")
    axes[i].set_xlabel("PDE Residual (m³/m³/s)")
    axes[i].legend(fontsize=7); light_grid(axes[i])
fig.suptitle(
    "Richards PDE Residual Distribution — 4-Model Benchmark  Device 108\n"
    "Smaller σ → physically consistent  |  PINN enforces this explicitly",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
savefig(fig, "figC8_pde_residual_4model.png")


# ── FIG C9 — Publication Table ───────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
ax.axis("off")
col_labels = ["Metric", "PINN\n(Richards)", "LSTM\n(seq input)",
              "Vanilla MLP", "Transient\nProxy"]

def mark_winner(vals, higher_better=True):
    best = max(vals) if higher_better else min(vals)
    return ["★" if abs(v - best) < 1e-9 else "" for v in vals]

rows = []
def add_row(label, key, fmt, higher_better=True, vals_override=None):
    vals = vals_override if vals_override is not None else \
           [bench[m][key] for m in MODEL_NAMES]
    wm   = mark_winner(vals, higher_better)
    rows.append([label] + [f"{v:{fmt}} {w}" for v, w in zip(vals, wm)])

add_row("Train R²",               "r2_tr",   ".4f", True)
add_row("Val R²",                 "r2_val",  ".4f", True)
add_row("Test R²  ★",            "r2_te",   ".4f", True)
add_row("Test RMSE (m³/m³)  ★", "rmse_te", ".5f", False)
add_row("Test MAE  (m³/m³)",     "mae_te",  ".5f", False)
add_row("PDE Residual σ  ★",     "pde_sd",  ".3e", False)
add_row("PDE Residual |μ|",      "pde_mu",  ".3e", False,
        vals_override=[abs(bench[m]["pde_mu"]) for m in MODEL_NAMES])
add_row("FoS Mean (test)",        "fos_mean",".3f", True)
add_row("FoS Min  ★",            "fos_min", ".3f", True)
add_row("FoS Std",                "fos_std", ".3f", False)
add_row("FoS<1.0 failures  ★",   "n_fail",  "d",   False)
add_row("FoS<1.3 warnings",       "n_warn",  "d",   False)
lat_vals = [bench[m]["lat_mean_ms"] for m in MODEL_NAMES]
wm = mark_winner(lat_vals, False)
rows.append(["Latency (ms)  ★"] +
            [f"{v:.3f} {w}" for v, w in zip(lat_vals, wm)])
spd_vals = [HYDRUS_LIT_SEC / (bench[m]["lat_mean_ms"]/1000) for m in MODEL_NAMES]
rows.append(["Speedup vs HYDRUS (×)"] + [f"{v:.0f}×" for v in spd_vals])
rows.append(["Physics constraint"] +
    ["✅ Richards PDE", "❌ None", "❌ None",
     "✅ Transient Euler"])          # updated label
param_vals = [
    sum(p.numel() for p in model.parameters()),
    sum(p.numel() for p in model_lstm.parameters()),
    sum(p.numel() for p in model_ml.parameters()),
    0,
]
rows.append(["Trainable params"] + [f"{v:,}" for v in param_vals])
# [FIX 3] λ selection row added
rows.append(["λ selection"] +
    [f"grid-search\n(λ={LAM_SELECTED})", "N/A", "N/A", "N/A"])

tbl = ax.table(cellText=rows, colLabels=col_labels,
               cellLoc="center", loc="center",
               colWidths=[0.28, 0.18, 0.18, 0.18, 0.18])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.2)
tbl.scale(1, 1.9)
header_bg = ["#023E8A", COLORS["PINN"], COLORS["LSTM"], COLORS["MLP"], COLORS["HYDRUS"]]
for j, hc in enumerate(header_bg):
    tbl[0, j].set_facecolor(hc)
    tbl[0, j].set_text_props(color="white", fontweight="bold")
highlight = {3, 4, 6, 9, 13}
for i in range(1, len(rows)+1):
    for j in range(5):
        cell_text = str(rows[i-1][j])
        if i in highlight:
            tbl[i,j].set_facecolor("#E3F2FD")
        elif i % 2 == 0:
            tbl[i,j].set_facecolor("#F8F9FA")
        else:
            tbl[i,j].set_facecolor("white")
        if "★" in cell_text and j > 0:
            tbl[i,j].set_facecolor("#D4EDDA")
ax.set_title(
    f"Table C — Comprehensive Benchmark: PINN · LSTM (seq) · Vanilla MLP · Transient Proxy\n"
    f"Device 108  ·  Sandy Clay Loam  ·  Slope {SLOPE:.0f}°  ·  "
    f"λ={LAM_SELECTED} (grid-searched)  ·  τ={TAU_SELECTED} (calibrated)  ·  "
    f"★ = winner per metric",
    fontsize=11, fontweight="bold", pad=16
)
plt.tight_layout()
savefig(fig, "figC9_benchmark_table.png")


# ── Console Summary ───────────────────────────────────────────
print("\n" + "="*70)
print("  4-MODEL BENCHMARK — FINAL RESULTS  Device 108")
print(f"  [FIX 1] LSTM: proper (N,15,2) sequence input")
print(f"  [FIX 2] HYDRUS: transient Euler proxy, τ={TAU_SELECTED} (calibrated)")
print(f"  [FIX 3] λ={LAM_SELECTED} (grid-searched, not hardcoded)")
print("="*70)
print(f"  {'Metric':<30} {'PINN':>9} {'LSTM':>9} {'MLP':>9} {'HYDRUS':>9}")
print("  " + "-"*60)
for label, key, hb, dec in [
    ("Train R²",            "r2_tr",      True,  4),
    ("Val   R²",            "r2_val",     True,  4),
    ("Test  R²  ★",        "r2_te",      True,  4),
    ("Test  RMSE  ★",      "rmse_te",    False, 5),
    ("PDE σ  ★",           "pde_sd",     False, 4),
    ("FoS Mean",            "fos_mean",   True,  3),
    ("FoS Min  ★",         "fos_min",    True,  3),
    ("FoS<1.0 failures  ★","n_fail",     False, 0),
    ("Latency (ms)  ★",    "lat_mean_ms",False, 3),
]:
    vals = [bench[m][key] for m in MODEL_NAMES]
    best = max(vals) if hb else min(vals)
    line = f"  {label:<30}"
    for v in vals:
        s = f"{int(v)}" if dec == 0 else f"{v:.{dec}f}"
        line += f" {s:>9}" + ("★" if abs(v-best)<1e-9 else " ")
    print(line)

print(f"\n  Figures → {OUT}/")
for name in ["figC1_loss_convergence_4model.png",
             "figC2_scatter_r2_4model.png",
             "figC3_theta_timeseries_4model.png",
             "figC4_fos_timeseries_4model.png",
             "figC5_metric_bars.png",
             "figC6_radar_4model.png",
             "figC7_speed_4model.png",
             "figC8_pde_residual_4model.png",
             "figC9_benchmark_table.png"]:
    print(f"   {name}")

████████████████████████████████████████████████████████████
  PRE-STEP: λ grid search (200 epoch × 5 candidates)
  This justifies LAM selection scientifically.
████████████████████████████████████████████████████████████
  λ =   0.5  →  best val MSE = 0.000037
  λ =   1.0  →  best val MSE = 0.000137
  λ =   5.0  →  best val MSE = 0.000304
  λ =  10.0  →  best val MSE = 0.000414
  λ =  50.0  →  best val MSE = 0.000684

  ★ Selected λ = 0.5  (val MSE = 0.000037)
  (Previously hardcoded LAM = 5; DIFFERENT — update LAM! )

  λ sensitivity table (for paper Methods section):
  λ          Best Val MSE
  0.5            0.000037 ← selected
  1.0            0.000137
  5.0            0.000304
  10.0           0.000414
  50.0           0.000684

  [FIX 1] LSTM input shape verified: (24766, 15, 2)  ✓
          (was: (N, 1, 16) — single timestep, recurrent memory unused)
          (now: (N, 15, 2)  — 15 timesteps, each with [rain, θ])

████████████████████████████████████████████████████████████
  

In [3]:
# ── FIG SA-3 — Tornado chart (±20% perturbation) ──────────────
fig, ax = plt.subplots(figsize=(10, 5))

# Baseline FoS_mean
baseline_fos_mean = ks_results[1.0]["FoS_mean"]

# Ks ±20%: find closest scales
ks_lo = ks_results[min(KS_SCALES, key=lambda s: abs(s - 0.80))]["FoS_mean"]
ks_hi = ks_results[min(KS_SCALES, key=lambda s: abs(s - 1.20))]["FoS_mean"]

# Slope ±20% in absolute degrees (33 ± ~6°)
sl_lo = slope_results[min(SLOPE_DEGS, key=lambda d: abs(d - 27.0))]["FoS_mean"]
sl_hi = slope_results[min(SLOPE_DEGS, key=lambda d: abs(d - 39.0))]["FoS_mean"]

params  = ["Hydraulic conductivity (Ks)", "Slope angle (β)"]
lo_vals = [ks_lo - baseline_fos_mean, sl_lo - baseline_fos_mean]
hi_vals = [ks_hi - baseline_fos_mean, sl_hi - baseline_fos_mean]

y_pos = np.arange(len(params))
bars_lo = ax.barh(y_pos, lo_vals, left=baseline_fos_mean,
                  color=C["phys"], alpha=0.8, label="−20% perturbation", height=0.4)
bars_hi = ax.barh(y_pos, hi_vals, left=baseline_fos_mean,
                  color=C["pred"], alpha=0.8, label="+20% perturbation", height=0.4)
ax.axvline(baseline_fos_mean, color="black", lw=1.5, ls="--",
           label=f"Baseline FoS_mean = {baseline_fos_mean:.3f}")
ax.axvline(1.0, color=C["warn"], lw=1.2, ls=":", label="FoS=1.0 ⚠")
ax.set_yticks(y_pos); ax.set_yticklabels(params, fontsize=11)
ax.set_xlabel("FoS Mean (−)"); light_grid(ax, axis="x")
ax.legend(fontsize=9)
ax.set_title("Tornado Chart — Parameter Sensitivity on Mean FoS  (±20% perturbation)\nDevice 108",
             fontweight="bold")

# Annotate values
for i, (lo, hi) in enumerate(zip(lo_vals, hi_vals)):
    # lo_val এর টেক্সট পজিশন (যদি মান খুব ছোট হয় তবে বারের বাইরে বামে দেখাবে)
    x_pos_lo = baseline_fos_mean + lo
    ha_lo = "right" if lo < -0.05 else "left"
    color_lo = "white" if lo < -0.05 else "black" # বারের বাইরে গেলে কালো টেক্সট
    offset_lo = -0.01 if lo < -0.05 else 0.01

    ax.text(x_pos_lo + offset_lo, i, f"{lo:+.3f}", ha=ha_lo,
            va="center", fontsize=9, color=color_lo, fontweight="bold")

    # hi_val এর টেক্সট পজিশন (যদি মান খুব ছোট হয় তবে বারের বাইরে ডানে দেখাবে)
    x_pos_hi = baseline_fos_mean + hi
    ha_hi = "left" if hi > 0.05 else "right"
    color_hi = "white" if hi > 0.05 else "black"
    offset_hi = 0.01 if hi > 0.05 else -0.01

    ax.text(x_pos_hi + offset_hi, i, f"{hi:+.3f}", ha=ha_hi,
            va="center", fontsize=9, color=color_hi, fontweight="bold")

plt.tight_layout()
savefig(fig, "figSA3_tornado_chart.png")

  → Saved: /content/piml_figures_108_pinn_richards/figSA3_tornado_chart.png


'/content/piml_figures_108_pinn_richards/figSA3_tornado_chart.png'